# Repte UPC Sostenible — Mode Difícil: entrenament, validació i exportació

Notebook **autocontingut**: funciona igual a **Kaggle**, a **Google Colab** i en local. Fa tot el pipeline:

1. Escriu els mòduls (`agent.py`, simulador, preprocessat, llibreria d'entrenament) a `hm_build/`.
2. Preprocessa `data/train` (PV ×12), tracta el forat de novembre 2025 i compara imputacions.
3. Construeix el **nostre conjunt de test**: 12 blocs de 168 h (la darrera setmana disponible de cada mes + novembre sintètic).
4. Entrena i **compara famílies de models** (seasonal-naive, Ridge, LightGBM, XGBoost, CatBoost, RandomForest) per MAE *i* per la puntuació oficial en Mode Difícil.
5. Ajusta amb **Optuna** els hiperparàmetres de LightGBM i els del control contra `CampusBatteryEnv` (6.000 kWh / ±1.500 kW / η 0,90 / horitzó 3 h).
6. Reentrena amb totes les dades, exporta `model_bateria.pkl` + `agent.py` i genera `submission.zip`.

**Dades** (els 6 CSV de `data/train/`, solts o dins d'un `.zip`): es localitzen automàticament.

* **Kaggle:** pugeu-los com a *Dataset* i afegiu-lo al notebook (`/kaggle/input/**`). No cal Internet.
* **Colab:** arrossegueu els CSV o un `data.zip` al panell de fitxers (`/content`), o deseu-los a Google Drive i poseu `MOUNT_DRIVE = True`. Si no es troben, el notebook obre un diàleg per pujar-los.
* **Local:** s'usa `../data/train` del repositori.

La GPU (CUDA) és opcional: es detecta i s'usa per a XGBoost/CatBoost. El resultat és `hm_build/submission.zip`.

In [ ]:
# ---------------- Configuració ----------------
DATA_DIR = None        # None = autodetecta (Kaggle, Colab, Drive, zip o repo local); o una ruta
MOUNT_DRIVE = False    # Colab: True per muntar Google Drive i buscar-hi les dades
FAST = False           # True = menys arbres i sense RandomForest/CatBoost (prova ràpida)
CHOSEN = "blend"       # model que s'exporta a l'agent: "blend" (LightGBM+Ridge), "lgb" o "ridge"
N_TRIALS_LGB = 20      # assaigs d'Optuna per objectiu (hiperparametres de LightGBM); 0 = no ajustar
N_TRIALS_CTRL = 60     # assaigs d'Optuna per als parametres del control; 0 = valors per defecte
FAMILIES = ["seasonal_naive", "ridge", "lightgbm_stacked", "lightgbm_buckets", "xgboost",
            "catboost", "random_forest"]
import os, sys, subprocess, zipfile
os.makedirs("hm_build", exist_ok=True)
IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content/sample_data")
IN_KAGGLE = os.path.isdir("/kaggle/input")
for pkg in ("lightgbm", "xgboost", "catboost", "optuna", "tqdm"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
warnings_off = os.environ.setdefault("PYTHONWARNINGS", "ignore")


def _walk(root, depth):
    # os.walk amb profunditat limitada (Drive pot ser enorme)
    root = os.path.abspath(root)
    base = root.rstrip(os.sep).count(os.sep)
    for d, dirs, files in os.walk(root):
        dirs[:] = [x for x in dirs if not x.startswith(".") and x not in ("hm_build", "sample_data", "__pycache__")]
        if d.count(os.sep) - base >= depth:
            dirs[:] = []
        yield d, files


def locate_data(data_dir=None):
    # Retorna la carpeta que conte contador.csv. Ordre: ruta explicita, carpetes conegudes,
    # un .zip amb els CSV (s'extreu a hm_data/), i a Colab un dialeg per pujar fitxers.
    need = "contador.csv"
    if data_dir and os.path.exists(os.path.join(data_dir, need)):
        return os.path.abspath(data_dir)
    if IN_COLAB and MOUNT_DRIVE and not os.path.isdir("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")
    roots = [("data/train", 1), ("../data/train", 1), ("hm_data", 4), (".", 3), ("..", 2),
             ("/kaggle/input", 5), ("/content", 3), ("/content/drive/MyDrive", 4)]
    roots = [(r, d) for r, d in roots if os.path.isdir(r)]
    zips = []
    for r, depth in roots:
        for d, files in _walk(r, depth):
            if need in files:
                return d
            zips += [os.path.join(d, f) for f in files if f.lower().endswith(".zip")]

    def unzip(path):
        try:
            with zipfile.ZipFile(path) as z:
                if any(os.path.basename(m) == need for m in z.namelist()):
                    z.extractall("hm_data")
                    return True
        except Exception:
            pass
        return False

    for zp in zips:
        if "submission" not in os.path.basename(zp).lower() and unzip(zp):
            return locate_data("hm_data") if os.path.exists("hm_data/" + need) else next(
                d for d, files in _walk("hm_data", 5) if need in files)
    if IN_COLAB:
        from google.colab import files
        print("No s'han trobat les dades: pugeu els 6 CSV de data/train (o un .zip que els contingui).")
        up = files.upload()
        os.makedirs("hm_data", exist_ok=True)
        for name in up:
            if name.lower().endswith(".zip"):
                unzip(name)
            else:
                os.replace(name, os.path.join("hm_data", os.path.basename(name)))
        for d, files_ in _walk("hm_data", 5):
            if need in files_:
                return d
    raise FileNotFoundError("No trobo contador.csv. Poseu DATA_DIR a la carpeta amb els CSV de data/train.")


DATA_DIR = locate_data(DATA_DIR)
missing = [f for f in ("contador.csv", "inversor.csv", "meteo.csv", "precio_pvpc.csv", "co2_red.csv")
           if not os.path.exists(os.path.join(DATA_DIR, f))]
assert not missing, f"Falten fitxers a {DATA_DIR}: {missing}"
print("Entorn:", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local", "| dades:", DATA_DIR)

## 1. Mòduls del projecte (s'escriuen a `hm_build/`)
`environment.py` és una còpia literal del simulador oficial de `starter_kit/`.

In [ ]:
%%writefile hm_build/environment.py
"""
starter_kit / environment.py
Entorn de simulació seqüencial pas a pas per al repte 'Carrega la Bateria!' (Datathon UPC Sostenible).
Avalua qualsevol BatteryAgent contra les dades del campus de forma rigorosa i sense data leakage.
"""

import json
import os
from pathlib import Path
import numpy as np
import pandas as pd

class CampusBatteryEnv:
    def __init__(self, hourly_data: pd.DataFrame, spec: dict = None, export_price: float = 0.0):
        """
        hourly_data: DataFrame indexat o amb columna 'Fecha' i columnes:
                     ['load_kw', 'pv_kw', 'precio_eur_per_kwh', 'co2_kg_per_kwh']
        spec: diccionari amb les especificacions físiques de la bateria.
        """
        if spec is None:
            spec_path = Path(__file__).parent / "battery_spec.json"
            with open(spec_path, encoding="utf-8") as f:
                spec = json.load(f)
                
        self.spec = spec
        self.export_price = export_price
        
        # Paràmetres de la bateria
        self.cap = float(spec["capacity_kwh"])
        self.power = float(spec["power_kw"])
        self.eta = float(spec["efficiency"])
        self.eta_c = self.eta ** 0.5
        self.eta_d = self.eta ** 0.5
        self.soc_init = float(spec["soc_init"]) * self.cap
        self.soc_min = float(spec["soc_min"]) * self.cap
        self.soc_max = float(spec["soc_max"]) * self.cap
        self.dt = float(spec.get("dt_hours", 1.0))
        
        self.df = hourly_data.copy().reset_index()
        if "Fecha" in self.df.columns:
            self.df["Fecha"] = pd.to_datetime(self.df["Fecha"])
        else:
            self.df["Fecha"] = pd.date_range("2026-01-01", periods=len(self.df), freq="1h")
            
        self.n = len(self.df)
        self.reset()

    def reset(self):
        self.t = 0
        self.soc = self.soc_init
        self.soc_series = []
        self.actions = []
        self.grid = []
        self.exports = []
        self.clipped = 0
        self.violations = 0
        self.infraccions_per_step = []
        return self._get_obs()

    def _get_obs(self):
        if self.t >= self.n:
            return None
            
        row = self.df.iloc[self.t]
        ts = row["Fecha"]
        
        # Previsions conegudes per a les properes 24 hores (PVPC i CO2)
        h_end = min(self.t + 24, self.n)
        future_prices = self.df["precio_eur_per_kwh"].iloc[self.t:h_end].values.tolist()
        future_co2 = self.df["co2_kg_per_kwh"].iloc[self.t:h_end].values.tolist()
        
        # Última hora transcorreguda (dades de t-1 a t)
        last_load = float(self.df["load_kw"].iloc[self.t - 1]) if self.t > 0 else float(row["load_kw"])
        last_pv = float(self.df["pv_kw"].iloc[self.t - 1]) if self.t > 0 else float(row["pv_kw"])
        
        obs = {
            "timestamp": ts,
            "hour": ts.hour,
            "day_of_week": ts.dayofweek,
            "soc_kwh": float(self.soc),
            "soc_pct": float(self.soc / self.cap),
            "soc_percent": float(self.soc / self.cap * 100),
            "preu_actual": float(row["precio_eur_per_kwh"]),
            "co2_actual": float(row["co2_kg_per_kwh"]),
            "preus_properes_24h": future_prices,
            "co2_properes_24h": future_co2,
            "noves_dades_5min": {
                "ultim_consum_kw": last_load,
                "ultima_solar_kw": last_pv
            }
        }
        return obs

    def step(self, action_kw: float):
        """Aplica l'acció en kW, valida la física i avança 1 hora."""
        p_req = float(action_kw)
        self.actions.append(p_req)
        infr_now = 0
        
        # 1. Limitar per potència màxima [-500, +500] kW
        p = np.clip(p_req, -self.power, self.power)
        if p != p_req:
            self.clipped += 1
            infr_now += 1

        # 2. Dinàmica del SOC
        if p > 0:  # Carregar
            max_c = (self.soc_max - self.soc) / (self.dt * self.eta_c)
            p_actual = min(p, max(0.0, max_c))
            if p_actual < p:
                self.clipped += 1
                infr_now += 1
            self.soc += p_actual * self.dt * self.eta_c
        elif p < 0:  # Descarregar
            max_d = (self.soc - self.soc_min) * self.eta_d / self.dt
            p_actual = -min(-p, max(0.0, max_d))
            if p_actual > p:
                self.clipped += 1
                infr_now += 1
            self.soc += p_actual * self.dt / self.eta_d
        else:
            p_actual = 0.0

        # Límits de seguretat del SOC
        if self.soc < self.soc_min - 1e-6 or self.soc > self.soc_max + 1e-6:
            self.violations += 1
            infr_now += 1
        self.soc = np.clip(self.soc, self.soc_min, self.soc_max)
        self.soc_series.append(self.soc)
        self.infraccions_per_step.append(infr_now)

        # 3. Balanç de xarxa amb el consum i la solar reals d'aquesta hora
        row = self.df.iloc[self.t]
        load = float(row["load_kw"])
        pv = float(row["pv_kw"])
        
        net = load - pv + p_actual
        if net > 0:
            self.grid.append(net)
            self.exports.append(0.0)
        else:
            self.grid.append(0.0)
            self.exports.append(-net)

        self.t += 1
        done = (self.t >= self.n)
        next_obs = None if done else self._get_obs()
        return next_obs, done

    def evaluate_results(self, alpha=0.7, beta=0.3, mask=None):
        """Calcula les mètriques finals comparant amb el Baseline sense bateria (opcionalment filtrades per una màscara temporal)."""
        grid = np.asarray(self.grid)
        exports = np.asarray(self.exports)
        price = self.df["precio_eur_per_kwh"].values[:len(grid)]
        co2 = self.df["co2_kg_per_kwh"].values[:len(grid)]

        # Baseline sense bateria: grid_base = max(0, load - pv)
        load = self.df["load_kw"].values[:len(grid)]
        pv = self.df["pv_kw"].values[:len(grid)]
        base_grid = np.maximum(load - pv, 0.0)

        if mask is not None:
            m = np.asarray(mask)[:len(grid)]
            infr_arr = np.asarray(self.infraccions_per_step)[:len(grid)]
            infraccions = int(np.sum(infr_arr[m]))
            grid = grid[m]
            exports = exports[m]
            price = price[m]
            co2 = co2[m]
            base_grid = base_grid[m]
        else:
            infraccions = int(self.clipped + self.violations)

        cost_participant = float(np.sum(grid * price) - np.sum(exports * self.export_price))
        co2_participant = float(np.sum(grid * co2))

        cost_base = float(np.sum(base_grid * price))
        co2_base = float(np.sum(base_grid * co2))

        d_co2 = (co2_base - co2_participant) / co2_base if co2_base > 0 else 0.0
        d_cost = (cost_base - cost_participant) / cost_base if cost_base > 0 else 0.0

        # Penalització per infraccions físiques (0.20 punts de Score per cada infracció)
        penalties = 0.002 * infraccions
        score = alpha * d_co2 + beta * d_cost - penalties

        return {
            "score": round(max(0.0, score * 100), 2),  # Punts sobre 100
            "co2_estalviat_pct": round(d_co2 * 100, 2),
            "cost_estalviat_pct": round(d_cost * 100, 2),
            "co2_estalviat_kg": round(co2_base - co2_participant, 1),
            "cost_estalviat_eur": round(cost_base - cost_participant, 2),
            "co2_participant_kg": round(co2_participant, 1),
            "cost_participant_eur": round(cost_participant, 2),
            "co2_baseline_kg": round(co2_base, 1),
            "cost_baseline_eur": round(cost_base, 2),
            "infraccions": infraccions,
        }

def run_agent_simulation(agent, hourly_data: pd.DataFrame, spec: dict = None):
    """Funció d'utilitat per executar un agent complet sobre una sèrie de dades."""
    env = CampusBatteryEnv(hourly_data, spec=spec)
    obs = env.reset()
    done = False
    while not done:
        action = agent.act(obs)
        obs, done = env.step(action)
    return env.evaluate_results()

In [ ]:
%%writefile hm_build/agent.py
"""
agent.py — Repte UPC Sostenible (Datathon FME 2026) — MODE DIFICIL
==================================================================
Bateria 6.000 kWh / +-1.500 kW / eta 0.90, solar x12, nomes 3 hores de preu i CO2.

Pipeline a cada pas t (tot offline, sense xarxa):
  1. `self.historial`: buffer incremental indexat per hora absoluta amb load, pv(x12),
     irradiancia, temperatures, preu i CO2. Es pre-carrega amb l'historic de train
     (les setmanes de test son la continuacio immediata de cada mes de train) i amb
     priors Oct/Des per al novembre 2025 (sense telemetria a train).
  2. Prediccio a 24 h de consum, solar, preu i CO2 (LightGBM multi-horitzo; Ridge de reserva).
     Preu/CO2: valors exactes per a k=0..2, model per a k=3..23.
  3. Planificacio de 24 passos amb importacio G_k = max(0, net_k + P_k) >= 0: absorbeix
     l'excedent solar gratuit i mai descarrega per sobre del consum net (export = 0 EUR).
     Dos solvers equivalents: LP (scipy.linprog / HiGHS) i DP sobre graella de SOC (numpy,
     el doble de rapid; per defecte). Es tria amb params["solver"].
  4. Clamp fisic exacte -> zero infraccions.

Tambe conte els constructors de features que fa servir train_pipeline.ipynb, de manera
que les features offline i online son identiques per construccio.
"""

import math
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import joblib
except Exception:  # pragma: no cover
    joblib = None
try:
    import lightgbm as lgb
except Exception:  # pragma: no cover
    lgb = None
try:
    from scipy.optimize import linprog
except Exception:  # pragma: no cover
    linprog = None
try:
    from scipy.special import ndtr
except Exception:  # pragma: no cover
    ndtr = None

_INV_SQRT_2PI = 0.3989422804014327


def expected_import(x, sigma):
    """E[max(x + e, 0)] amb e ~ N(0, sigma): importacio esperada quan el consum net previst
    es x i l'error de prediccio te desviacio sigma. Suavitza el colze de max(x, 0)."""
    z = x / sigma
    return x * ndtr(z) + sigma * _INV_SQRT_2PI * np.exp(-0.5 * z * z)


ARTIFACT_NAME = "model_bateria.pkl"

# ----------------------------------------------------------------------------------
# Eix temporal absolut (hores des de EPOCH)
# ----------------------------------------------------------------------------------
EPOCH = pd.Timestamp("2025-08-01 00:00:00")
END = pd.Timestamp("2027-01-01 00:00:00")
_EPOCH_H = EPOCH.value // 3_600_000_000_000
N_HOURS = int((END - EPOCH) // pd.Timedelta(hours=1))
SERIES = ("load", "pv", "irr", "tamb", "tpan", "price", "co2")
TELEMETRY = ("load", "pv", "irr", "tamb", "tpan")
HORIZON = 24
NAN = float("nan")


def hour_index(ts) -> int:
    """Hora absoluta (enter) d'un timestamp respecte EPOCH."""
    return int(pd.Timestamp(ts).value // 3_600_000_000_000 - _EPOCH_H)


def full_index() -> pd.DatetimeIndex:
    return pd.date_range(EPOCH, periods=N_HOURS, freq="1h")


# ----------------------------------------------------------------------------------
# Calendari Barcelona / UPC (nomes depen del timestamp)
# ----------------------------------------------------------------------------------
BCN_HOLIDAYS = [
    "2025-08-15", "2025-09-11", "2025-09-24", "2025-10-12", "2025-11-01", "2025-12-06",
    "2025-12-08", "2025-12-25", "2025-12-26",
    "2026-01-01", "2026-01-06", "2026-04-03", "2026-04-06", "2026-05-01", "2026-05-25",
    "2026-06-24", "2026-08-15", "2026-09-11", "2026-09-24", "2026-10-12", "2026-11-01",
    "2026-12-06", "2026-12-08", "2026-12-25", "2026-12-26",
]
TERM_BREAKS = [("2025-12-24", "2026-01-06"), ("2026-03-30", "2026-04-06"),
               ("2026-12-24", "2027-01-06")]
CAL_NAMES = ["hour", "dow", "h_sin", "h_cos", "d_sin", "d_cos", "y_sin", "y_cos",
             "is_weekend", "is_bcn_holiday", "is_august_recess", "is_exam_period",
             "is_datathon_weekend", "is_term_break", "is_offday"]
C_HSIN, C_HCOS = 2, 3


def calendar_matrix(index: pd.DatetimeIndex) -> np.ndarray:
    """Features ciclics + flags is_x, calculables nomes amb el timestamp."""
    hour = index.hour.values.astype(float)
    dow = index.dayofweek.values.astype(float)
    doy = index.dayofyear.values.astype(float)
    day = index.normalize()
    month = index.month.values
    dom = index.day.values
    out = np.zeros((len(index), len(CAL_NAMES)))
    out[:, 0] = hour
    out[:, 1] = dow
    out[:, 2] = np.sin(2 * np.pi * hour / 24.0)
    out[:, 3] = np.cos(2 * np.pi * hour / 24.0)
    out[:, 4] = np.sin(2 * np.pi * dow / 7.0)
    out[:, 5] = np.cos(2 * np.pi * dow / 7.0)
    out[:, 6] = np.sin(2 * np.pi * doy / 365.25)
    out[:, 7] = np.cos(2 * np.pi * doy / 365.25)
    out[:, 8] = dow >= 5
    out[:, 9] = day.isin(pd.to_datetime(BCN_HOLIDAYS))
    out[:, 10] = month == 8
    exam = ((month == 1) & (dom >= 8) & (dom <= 23)) | ((month == 5) & (dom >= 25)) | \
           ((month == 6) & (dom <= 20))
    out[:, 11] = exam
    out[:, 12] = (index.year.values == 2026) & (month == 11) & ((dom == 14) | (dom == 15))
    brk = np.zeros(len(index), dtype=bool)
    for a, b in TERM_BREAKS:
        brk |= (day >= pd.Timestamp(a)) & (day <= pd.Timestamp(b))
    out[:, 13] = brk
    out[:, 14] = (out[:, 8] > 0) | (out[:, 9] > 0) | brk
    return out


# ----------------------------------------------------------------------------------
# Constructors de features (compartits offline / online)
# ----------------------------------------------------------------------------------
def _stats(x):
    """mean, std, min, max ignorant NaN (sense warnings)."""
    v = x[x == x]
    if v.size == 0:
        return NAN, NAN, NAN, NAN
    return float(v.mean()), float(v.std()), float(v.min()), float(v.max())


def _mean(x):
    v = x[x == x]
    return float(v.mean()) if v.size else NAN


_K = np.arange(HORIZON)
_CALN = len(CAL_NAMES)

LOAD_FEATS = (["k"] + ["cal_" + c for c in CAL_NAMES] + ["t_hsin", "t_hcos"] +
              ["l1", "l2", "l3", "l_h24", "l_h48", "l_h168", "l_t25",
               "l_m24", "l_s24", "l_min24", "l_max24", "l_m168", "tamb1", "irr1", "pv1", "l_adj"])
PV_FEATS = (["k"] + ["cal_" + c for c in CAL_NAMES] + ["t_hsin", "t_hcos"] +
            ["pv1", "pv2", "pv3", "irr1", "irr2", "tpan1", "tamb1", "pv_h24", "pv_h48",
             "pv_h168", "irr_h24", "cs_h", "cs_t1", "kt1", "kt_day", "kt_h24",
             "pv_m24", "pv_max24", "pv_pers", "pv_day"])
MKT_FEATS = (["k"] + ["cal_" + c for c in CAL_NAMES] + ["t_hsin", "t_hcos"] +
             ["y0", "y1", "y2", "ym1", "y_h24", "y_h48", "y_h168", "d0", "d2",
              "y_m24", "y_s24", "y_min24", "y_max24", "y_m168",
              "o0", "o2", "o_h24", "irr1", "kt_day", "y_adj", "ym3", "ym6", "ym12", "slope",
              "o_slope"])


class Features:
    """Construeix les matrius (24, F) per a les hores objectiu t..t+23.

    Regla anti-leakage (verificada per tests): la telemetria (load, pv, meteo) nomes
    s'indexa fins a t-1 i el mercat (preu, CO2) nomes fins a t+2.
    """

    def __init__(self, hist: dict, cal: np.ndarray, cs: np.ndarray):
        self.h = hist      # dict nom -> array (N_HOURS,)
        self.cal = cal     # (N_HOURS, len(CAL_NAMES))
        self.cs = cs       # (N_HOURS,) envolupant solar de cel clar (kW, x12)

    def _base(self, t, n_extra):
        X = np.empty((HORIZON, 1 + _CALN + 2 + n_extra))
        X[:, 0] = _K
        X[:, 1:1 + _CALN] = self.cal[t:t + HORIZON]
        X[:, 1 + _CALN] = self.cal[t, C_HSIN]
        X[:, 2 + _CALN] = self.cal[t, C_HCOS]
        return X, 3 + _CALN

    def _kt_day(self, t):
        pv = self.h["pv"][t - 24:t]
        cs = self.cs[t - 24:t]
        m = pv == pv
        den = float(cs[m].sum()) if m.any() else 0.0
        return float(pv[m].sum()) / den if den > 100.0 else NAN

    def load(self, t):
        L = self.h["load"]
        X, j = self._base(t, 16)
        X[:, j] = L[t - 1]; X[:, j + 1] = L[t - 2]; X[:, j + 2] = L[t - 3]
        X[:, j + 3] = L[t - 24:t]
        X[:, j + 4] = L[t - 48:t - 24]
        X[:, j + 5] = L[t - 168:t - 144]
        X[:, j + 6] = L[t - 25]
        X[:, j + 7:j + 11] = _stats(L[t - 24:t])
        X[:, j + 11] = _mean(L[t - 168:t])
        X[:, j + 12] = self.h["tamb"][t - 1]
        X[:, j + 13] = self.h["irr"][t - 1]
        X[:, j + 14] = self.h["pv"][t - 1]
        X[:, j + 15] = L[t - 1] + L[t - 24:t] - L[t - 25]      # persistencia + forma d'ahir
        return X

    def pv(self, t):
        P = self.h["pv"]; I = self.h["irr"]; cs = self.cs
        X, j = self._base(t, 20)
        X[:, j] = P[t - 1]; X[:, j + 1] = P[t - 2]; X[:, j + 2] = P[t - 3]
        X[:, j + 3] = I[t - 1]; X[:, j + 4] = I[t - 2]
        X[:, j + 5] = self.h["tpan"][t - 1]
        X[:, j + 6] = self.h["tamb"][t - 1]
        X[:, j + 7] = P[t - 24:t]
        X[:, j + 8] = P[t - 48:t - 24]
        X[:, j + 9] = P[t - 168:t - 144]
        X[:, j + 10] = I[t - 24:t]
        X[:, j + 11] = cs[t:t + HORIZON]
        X[:, j + 12] = cs[t - 1]
        kt1 = P[t - 1] / cs[t - 1] if cs[t - 1] > 30.0 else NAN
        ktd = self._kt_day(t)
        X[:, j + 13] = kt1
        X[:, j + 14] = ktd
        X[:, j + 18] = (kt1 if kt1 == kt1 else ktd) * cs[t:t + HORIZON]   # persistencia de claredat
        X[:, j + 19] = ktd * cs[t:t + HORIZON]
        csy = cs[t - 24:t]
        with np.errstate(divide="ignore", invalid="ignore"):
            X[:, j + 15] = np.where(csy > 30.0, P[t - 24:t] / csy, NAN)
        st = _stats(P[t - 24:t])
        X[:, j + 16] = st[0]; X[:, j + 17] = st[3]
        return X

    def market(self, t, own, other):
        Y = self.h[own]; O = self.h[other]
        X, j = self._base(t, 25)
        X[:, j] = Y[t]; X[:, j + 1] = Y[t + 1]; X[:, j + 2] = Y[t + 2]; X[:, j + 3] = Y[t - 1]
        X[:, j + 4] = Y[t - 24:t]
        X[:, j + 5] = Y[t - 48:t - 24]
        X[:, j + 6] = Y[t - 168:t - 144]
        X[:, j + 7] = Y[t] - Y[t - 24]
        X[:, j + 8] = Y[t + 2] - Y[t - 22]
        X[:, j + 9:j + 13] = _stats(Y[t - 21:t + 3])
        X[:, j + 13] = _mean(Y[t - 165:t + 3])
        X[:, j + 14] = O[t]; X[:, j + 15] = O[t + 2]
        X[:, j + 16] = O[t - 24:t]
        X[:, j + 17] = self.h["irr"][t - 1]
        X[:, j + 18] = self._kt_day(t)
        X[:, j + 19] = Y[t + 2] + Y[t - 24:t] - Y[t - 22]       # ancoratge a l'ultim valor exacte
        X[:, j + 20] = Y[t - 3]; X[:, j + 21] = Y[t - 6]; X[:, j + 22] = Y[t - 12]
        X[:, j + 23] = Y[t + 2] - Y[t]
        X[:, j + 24] = O[t + 2] - O[t]
        return X

    def build(self, target, t):
        if target == "load":
            return self.load(t)
        if target == "pv":
            return self.pv(t)
        if target == "price":
            return self.market(t, "price", "co2")
        return self.market(t, "co2", "price")


FEATURE_NAMES = {"load": LOAD_FEATS, "pv": PV_FEATS, "price": MKT_FEATS, "co2": MKT_FEATS}


def clear_sky_series(cs_table: np.ndarray, index: pd.DatetimeIndex) -> np.ndarray:
    """cs_table (367, 24) -> serie horaria sobre `index`."""
    return cs_table[index.dayofyear.values, index.hour.values]


def seed_gap_priors(hist: dict, cal: np.ndarray):
    """Priors de cold-start a `self.historial`:
    * Novembre 2025 (sense comptador / inversor / meteo a train): perfil horari per tipus
      de dia (lectiu / no lectiu) del periode academic Oct + 1-21 Des 2025.
    * Solar de finals d'agost 2026 (l'inversor s'atura el 16/08): perfil horari d'agost.
    Nomes s'omplen hores sense dada; les mesures reals sobreescriuen el prior."""
    i = hour_index
    nov0, nov1 = i("2025-11-01"), i("2025-12-01")
    src = np.zeros(N_HOURS, dtype=bool)
    src[i("2025-10-01"):i("2025-11-01")] = True
    src[i("2025-12-01"):i("2025-12-22")] = True
    hour = cal[:, 0].astype(int)
    off = cal[:, 14] > 0
    for name in TELEMETRY:
        a = hist[name]
        if not np.isnan(a[nov0:nov1]).any():
            continue
        for o in (False, True):
            for h in range(24):
                m = src & (hour == h) & (off == o) & ~np.isnan(a)
                if not m.any():
                    m = src & (hour == h) & ~np.isnan(a)
                if not m.any():
                    continue
                tgt = np.zeros(N_HOURS, dtype=bool)
                tgt[nov0:nov1] = True
                tgt &= (hour == h) & (off == o) & np.isnan(a)
                a[tgt] = a[m].mean()
    a0, a1 = i("2026-08-01"), i("2026-09-01")
    for name in ("pv",):
        a = hist[name]
        for h in range(24):
            m = np.zeros(N_HOURS, dtype=bool)
            m[a0:a1] = True
            m &= (hour == h)
            known = m & ~np.isnan(a)
            if known.any():
                a[m & np.isnan(a)] = a[known].mean()


# ----------------------------------------------------------------------------------
# Models de prediccio
# ----------------------------------------------------------------------------------
# Grups d'horitzo k amb LightGBM. Mes enlla de k=11 nomes Ridge (mateixa precisio, 5x mes rapid).
BUCKETS = ((0, 0), (1, 1), (2, 2), (3, 5), (6, 11))


class Forecaster:
    """Un objectiu = LightGBM per grup d'horitzo (k <= 11) + Ridge directe per horitzo.
    prefer="blend": mitjana LightGBM/Ridge on hi ha tots dos; "lgb": LightGBM i Ridge nomes
    on no hi ha arbres; "ridge": nomes Ridge (numpy pur, sense dependencies)."""

    def __init__(self, spec: dict, prefer: str = "blend", blend_w: float = 0.5):
        self.boosters = []
        self.has_ridge = False
        self.blend = float(blend_w) if prefer == "blend" else 0.0
        if prefer in ("lgb", "blend") and lgb is not None and spec.get("lgb"):
            try:
                for (k0, k1, txt) in spec["lgb"]:
                    self.boosters.append((int(k0), int(k1) + 1, lgb.Booster(model_str=txt)))
            except Exception:
                self.boosters = []
        if spec.get("ridge") is not None:
            r = spec["ridge"]
            self.W = np.asarray(r["W"], dtype=float)     # (24, F)
            self.b = np.asarray(r["b"], dtype=float)     # (24,)
            self.mu = np.asarray(r["mu"], dtype=float)
            self.sd = np.asarray(r["sd"], dtype=float)
            self.has_r = np.abs(self.W).sum(axis=1) > 0
            self.has_ridge = True
        self.kind = "lgb" if self.boosters else ("ridge" if self.has_ridge else None)

    def predict(self, X):
        """X: (24, F) -> prediccio (24,). Les k sense cap model queden a NaN."""
        out = np.full(X.shape[0], NAN)
        for k0, k1, bst in self.boosters:
            out[k0:k1] = bst.predict(X[k0:k1], num_threads=1)
        if self.has_ridge:
            Z = (np.where(X == X, X, self.mu) - self.mu) / self.sd
            r = np.where(self.has_r, (Z * self.W).sum(axis=1) + self.b, NAN)
            mix = np.where(r == r, (1.0 - self.blend) * out + self.blend * r, out)
            out = np.where(out == out, mix, r)
        return out


DEFAULT_PARAMS = {
    "lam_term": 0.95,          # V_term = lam * mean(w previst) per kWh emmagatzemat al final
    "lam_weekend": 1.0,        # factor multiplicatiu si l'hora terminal cau en dia no lectiu
    "lam_month": [1.0] * 12,   # factor per mes (regim estacional)
    "cap_q": 0.2,              # quantil (consum baix / solar alta) per al sostre de descarrega; 0 = off
    "buf_kw": 0.0,             # marge de seguretat d'exportacio a k=0 (kW)
    "buf_frac": 0.0,           # marge relatiu sobre el consum net previst
    "gamma": 1.0,              # descompte d'incertesa per a k>=3 (1 = sense descompte)
    "eps_thr": 1e-5,           # penalitzacio minima de throughput (evita carrega+descarrega)
    "co2_ref": 0.085,          # CO2 mitja ponderada del baseline (kg/kWh)
    "price_ref": 0.139,        # preu mitja ponderat del baseline (EUR/kWh)
    "sim_end": "2026-08-31 23:00:00",   # ultima hora del test: mes enlla el SOC no val res
    "horizon": 24,             # hores planificades (24..48); k>=24 = prediccio de k-24 esmorteida
    "gamma_far": 0.7,          # esmorteiment cap a la mitjana dels pesos w per a k>=24
    "w_co2": 0.7,              # pes del CO2 a l'objectiu del planificador (metrica oficial: 0.7)
    "sigma_steps": 6,          # nombre de passos (des de k=0) amb cost estocastic
    "sigma_scale": 0.6,        # >0: DP estocastic (multiplica la sigma d'error del consum net)
    "net_bias_kw": 0.0,        # biaix afegit al consum net previst per a k>=1 (kW)
    "solver": "dp",            # "dp" (graella de SOC, numpy) o "lp" (scipy.linprog / HiGHS)
    "dp_grid": 81,
    "prefer": "blend",         # "blend" (LightGBM+Ridge), "lgb" o "ridge"
}


class BatteryAgent:
    def __init__(self, artifact=None):
        """`artifact`: dict (per a validacio) o None -> carrega model_bateria.pkl del costat."""
        self.ready = False
        self.params = dict(DEFAULT_PARAMS)
        self.prev_idx = None
        self.cap = 6000.0
        self.pmax = 1500.0
        self.eta = 0.90 ** 0.5
        self.last = {}
        self.dropout = False
        self.net_lo = None
        self.n_fallback = 0        # passos resolts amb l'heuristica de reserva (diagnostic)
        try:
            if artifact is None:
                artifact = joblib.load(Path(__file__).parent / ARTIFACT_NAME)
            self._setup(artifact)
            self.ready = True
        except Exception:
            self.ready = False

    # ------------------------------------------------------------------ init
    def _setup(self, art):
        self.params.update(art.get("params", {}))
        idx = full_index()
        self.cal = calendar_matrix(idx)
        # Buffer d'estat incremental: historial[serie][hora_absoluta]
        self.historial = {}
        for name in SERIES:
            a = np.full(N_HOURS, NAN)
            src = np.asarray(art["hist"][name], dtype=float)
            a[:len(src)] = src
            self.historial[name] = a
        seed_gap_priors(self.historial, self.cal)
        self.cs = clear_sky_series(np.asarray(art["cs_table"], dtype=float), idx)
        self.feat = Features(self.historial, self.cal, self.cs)
        bw = self.params.get("blend_w", 0.5)
        if not isinstance(bw, dict):
            bw = {k: bw for k in ("load", "pv", "price", "co2")}
        self.models = {k: Forecaster(art["models"][k], self.params.get("prefer", "blend"),
                                     bw.get(k, 0.5))
                       for k in ("load", "pv", "price", "co2")}
        # Models quantil de k=0 per al sostre de descarrega (evita exportar energia a 0 EUR)
        self.cap_models = None
        try:
            cm = (art["models"].get("cap") or {}).get(float(self.params.get("cap_q", 0.0)))
            if cm and lgb is not None and self.params.get("prefer", "lgb") != "ridge":
                self.cap_models = (lgb.Booster(model_str=cm[0]), lgb.Booster(model_str=cm[1]))
        except Exception:
            self.cap_models = None
        self.irr_per_pv = float(art.get("irr_per_pv", 0.6))
        sg = art.get("sigma") or {}
        self.sig_load = np.asarray(sg["load"], dtype=float) if "load" in sg else None
        self.sig_pv = np.asarray(sg["pv_frac"], dtype=float) if "pv_frac" in sg else None
        if self.sig_pv is None:
            self.sig_load = None
        self.sim_end_idx = hour_index(self.params["sim_end"])
        self._lp_cache = {}

    # ------------------------------------------------------------------ estat
    def _update_history(self, t, obs):
        H = self.historial
        prices = [float(v) for v in obs.get("preus_properes_24h", [])][:HORIZON]
        co2s = [float(v) for v in obs.get("co2_properes_24h", [])][:HORIZON]
        if not prices:
            prices = [float(obs["preu_actual"])]
        if not co2s:
            co2s = [float(obs["co2_actual"])]
        prices[0] = float(obs.get("preu_actual", prices[0]))
        co2s[0] = float(obs.get("co2_actual", co2s[0]))
        m = min(len(prices), len(co2s))
        prices, co2s = prices[:m], co2s[:m]
        # No desem a l'historial valors mes enlla d'una frontera de bloc (canvi de mes o
        # inici dels darrers 7 dies): la fila seguent de l'entorn pot ser d'una altra data.
        ts = EPOCH + pd.Timedelta(hours=t)
        for k in range(m):
            tk = ts + pd.Timedelta(hours=k)
            if k > 0 and (tk.month != ts.month or
                          (tk.day > tk.days_in_month - 7) != (ts.day > ts.days_in_month - 7)):
                break
            if prices[k] == prices[k]:
                H["price"][t + k] = prices[k]
            if co2s[k] == co2s[k]:
                H["co2"][t + k] = co2s[k]

        # Telemetria de la darrera hora: correspon a la fila anterior de l'entorn (prev_idx).
        nd = obs.get("noves_dades_5min") or {}
        p = self.prev_idx
        self.dropout = False
        if p is not None and p != t and 0 <= p < N_HOURS:
            def val(key):
                v = nd.get(key)
                try:
                    v = float(v)
                except Exception:
                    return NAN
                return v
            load = val("ultim_consum_kw")
            pv = val("ultima_solar_kw")
            if load == load:
                if load > 1.0:
                    H["load"][p] = load
                else:
                    self.dropout = True          # 0.0 = caiguda del sensor del comptador
            if pv == pv:
                H["pv"][p] = max(0.0, pv)
            irr = val("irradiancia_wm2")
            if irr != irr and pv == pv:
                irr = max(0.0, pv) * self.irr_per_pv    # r(pv, irradiancia) = 0.997
            if irr == irr:
                H["irr"][p] = max(0.0, irr)
            ta = val("temp_ambient_c")
            if ta == ta:
                H["tamb"][p] = ta
            elif H["tamb"][p] != H["tamb"][p]:
                H["tamb"][p] = H["tamb"][p - 1]
            tp = val("temp_panel_c")
            if tp == tp:
                H["tpan"][p] = tp
            elif H["tpan"][p] != H["tpan"][p]:
                H["tpan"][p] = H["tpan"][p - 1]
        return prices, co2s

    # ------------------------------------------------------------------ prediccio
    def forecast(self, t, prices, co2s):
        f = self.feat
        Xl = f.load(t)
        Xp = f.pv(t)
        load, pv, price, co2 = self._predict_all(t, Xl, Xp)
        cs = self.cs[t:t + HORIZON]
        self.net_lo = None
        if self.cap_models is not None:
            lo = float(self.cap_models[0].predict(Xl[:1], num_threads=1)[0])
            hi = float(self.cap_models[1].predict(Xp[:1], num_threads=1)[0])
            hi = min(max(hi, 0.0), 1.2 * cs[0] + 20.0) if cs[0] > 1.0 else 0.0
            self.net_lo = lo - hi
        pv = np.where(cs > 1.0, np.clip(pv, 0.0, 1.2 * cs + 20.0), 0.0)
        load = np.maximum(load, 0.0)
        m = len(prices)
        price[:m] = prices
        co2[:m] = co2s
        if price[-1] != price[-1] or co2[-1] != co2[-1]:      # reserva: persistencia
            price = np.where(price == price, price, prices[-1])
            co2 = np.where(co2 == co2, co2, co2s[-1])
        price = np.maximum(price, 0.0)
        co2 = np.maximum(co2, 0.0)
        return load, pv, price, co2

    def _predict_all(self, t, Xl, Xp):
        """Prediccions crues (24,) de consum, solar, preu i CO2."""
        f = self.feat
        return (self.models["load"].predict(Xl), self.models["pv"].predict(Xp),
                self.models["price"].predict(f.market(t, "price", "co2")),
                self.models["co2"].predict(f.market(t, "co2", "price")))

    # ------------------------------------------------------------------ LP
    def _lp_matrices(self, n):
        if n in self._lp_cache:
            return self._lp_cache[n]
        I = np.eye(n)
        T = np.tril(np.ones((n, n)))
        A = np.zeros((3 * n, 3 * n))
        A[:n, :n] = I; A[:n, n:2 * n] = -I; A[:n, 2 * n:] = -I          # c - d - g <= -net
        A[n:2 * n, :n] = T * self.eta; A[n:2 * n, n:2 * n] = -T / self.eta     # soc <= cap
        A[2 * n:, :n] = -T * self.eta; A[2 * n:, n:2 * n] = T / self.eta       # soc >= 0
        self._lp_cache[n] = A
        return A

    def plan(self, soc, net, w, v_term, d_cap0):
        """LP de n passos. Variables: carrega c_k, descarrega d_k, importacio g_k >= 0."""
        n = len(net)
        A = self._lp_matrices(n)
        P = self.params
        cost = np.empty(3 * n)
        cost[:n] = P["eps_thr"] - v_term * self.eta
        cost[n:2 * n] = P["eps_thr"] + v_term / self.eta
        cost[2 * n:] = w
        b = np.empty(3 * n)
        b[:n] = -net
        b[n:2 * n] = self.cap - soc
        b[2 * n:] = soc
        ub = np.full(3 * n, np.inf)
        ub[:2 * n] = self.pmax
        ub[n] = min(self.pmax, d_cap0)
        res = linprog(cost, A_ub=A, b_ub=b, bounds=np.column_stack((np.zeros(3 * n), ub)),
                      method="highs")
        if res.status != 0:
            return None
        return float(res.x[0] - res.x[n])

    def plan_dp(self, soc, net, w, v_term, d_cap0, sigma=None):
        """Mateix problema que `plan`, resolt per programacio dinamica sobre una graella de
        SOC (numpy). Si es dona `sigma` (desviacio de l'error del consum net per pas), el cost
        de cada pas es la importacio ESPERADA w_k * E[max(net_k + e_k + P, 0)]: control
        estocastic conscient de l'error de prediccio (decideix quant arriscar-se a exportar
        en funcio del valor de l'energia a cada hora)."""
        n = len(net)
        stoch = sigma is not None and ndtr is not None
        G = int(self.params.get("dp_grid", 81))
        key = ("dp", G, self.cap, self.pmax)
        if key not in self._lp_cache:
            grid = np.linspace(0.0, self.cap, G)
            D = grid[None, :] - grid[:, None]                    # s -> s'
            Pm = np.where(D >= 0, D / self.eta, D * self.eta)    # potencia necessaria (kW)
            pen = np.where(np.abs(Pm) <= self.pmax + 1e-9, 0.0, np.inf)
            self._lp_cache[key] = (grid, Pm, pen)
        grid, Pm, pen = self._lp_cache[key]
        eps = self.params["eps_thr"]
        V = -v_term * grid
        for k in range(n - 1, 0, -1):
            if stoch and sigma[k] > 1.0:
                C = expected_import(Pm + net[k], sigma[k])
            else:
                C = np.maximum(Pm + net[k], 0.0)
            C *= w[k]
            C += pen
            C += V[None, :]
            V = C.min(axis=1)
        lo = -min(self.pmax, soc * self.eta, d_cap0)
        hi = min(self.pmax, (self.cap - soc) / self.eta)
        ds = grid - soc
        cand = np.concatenate((np.where(ds >= 0, ds / self.eta, ds * self.eta),
                               (0.0, -net[0], lo, hi)))
        if stoch and sigma[0] > 1.0:
            cand = np.concatenate((cand, np.linspace(lo, hi, 61)))
        cand = np.clip(cand, lo, hi)
        s1 = soc + np.where(cand >= 0, cand * self.eta, cand / self.eta)
        if stoch and sigma[0] > 1.0:
            imp0 = expected_import(net[0] + cand, sigma[0])
        else:
            imp0 = np.maximum(net[0] + cand, 0.0)
        val = w[0] * imp0 + np.interp(s1, grid, V) + eps * np.abs(cand)
        return float(cand[int(np.argmin(val))])

    def decide(self, t, soc, prices, co2s):
        P = self.params
        load, pv, price, co2 = self.forecast(t, prices, co2s)
        if self.dropout:
            load[0] = 0.0
        net = load - pv
        if P["net_bias_kw"] != 0.0:
            net[1:] += P["net_bias_kw"]
        a_co2 = float(P["w_co2"])
        w = a_co2 * co2 / P["co2_ref"] + (1.0 - a_co2) * price / P["price_ref"]
        m = len(prices)
        if P["gamma"] != 1.0 and m < HORIZON:
            wbar = float(w.mean())
            w[m:] = wbar + P["gamma"] * (w[m:] - wbar)
        # Horitzo ampliat (fins a 48 h): les hores k>=24 repeteixen la prediccio de k-24
        # (mateixa hora de l'endema) amb els pesos esmorteits cap a la mitjana.
        Hn = int(min(max(P.get("horizon", HORIZON), HORIZON), 2 * HORIZON))
        if Hn > HORIZON:
            net, w = self._extend_horizon(t, net, w, Hn)
        # Horitzo retallat al final conegut de la simulacio.
        n = Hn
        remaining = self.sim_end_idx - t + 1
        v_scale = 1.0
        if 0 < remaining <= Hn:
            n = max(1, remaining)
            v_scale = 0.0
        if m < 3:                      # l'entorn s'esta acabant (llistes retallades)
            n = max(1, m)
            v_scale = 0.0
        te = min(t + min(n, HORIZON), N_HOURS - 1)
        lam = P["lam_term"] * P["lam_month"][int(self.month[te]) - 1]
        if self.cal[te, 14] > 0:
            lam *= P["lam_weekend"]
        v_term = v_scale * lam * float(w[:HORIZON].mean())
        net_lo = net[0]
        if self.net_lo is not None and not self.dropout:
            net_lo = min(net_lo, self.net_lo)
        d_cap0 = max(0.0, net_lo * (1.0 - P["buf_frac"]) - P["buf_kw"])
        self.last = {"load": load, "pv": pv, "price": price, "co2": co2, "w": w}
        a = None
        if self.params.get("solver", "dp") == "lp" and linprog is not None:
            try:
                a = self.plan(soc, net[:n], w[:n], v_term, d_cap0)
            except Exception:
                a = None
        if a is None or a != a:
            a = self.plan_dp(soc, net[:n], w[:n], v_term, d_cap0, self._sigma(t, n))
        return a

    def _sigma(self, t, n):
        """Desviacio de l'error del consum net per pas: sqrt(sig_load^2 + (sig_pv * cs)^2)."""
        sc = float(self.params.get("sigma_scale", 0.0))
        if sc <= 0.0 or self.sig_load is None:
            return None
        k = np.minimum(np.arange(n), HORIZON - 1)
        cs = self.cs[t:t + n]
        if len(cs) < n:
            return None
        s = np.sqrt(self.sig_load[k] ** 2 + (self.sig_pv[k] * cs) ** 2) * sc
        s[int(self.params.get("sigma_steps", 6)):] = 0.0     # mes enlla: cost determinista
        if self.dropout:
            s[0] = 0.0
        return s

    def _extend_horizon(self, t, net, w, Hn):
        extra = Hn - HORIZON
        wbar = float(w.mean())
        w2 = wbar + self.params["gamma_far"] * (w[:extra] - wbar)
        return np.concatenate((net, net[:extra])), np.concatenate((w, w2))

    @property
    def month(self):
        mth = getattr(self, "_month", None)
        if mth is None:
            mth = full_index().month.values
            self._month = mth
        return mth

    # ------------------------------------------------------------------ fallback
    @staticmethod
    def _heuristic(obs):
        hora = int(obs.get("hour", 0))
        nd = obs.get("noves_dades_5min") or {}
        try:
            net = float(nd.get("ultim_consum_kw", 0.0)) - float(nd.get("ultima_solar_kw", 0.0))
        except Exception:
            net = 0.0
        if 10 <= hora <= 15:
            return 1000.0
        if 18 <= hora <= 22:
            return -max(0.0, 0.8 * net)
        return 0.0

    # ------------------------------------------------------------------ act
    def act(self, obs: dict) -> float:
        consigna_desitjada = 0.0
        try:
            soc_kwh = float(obs["soc_kwh"])
            cap = obs.get("cap_max_kwh")
            if cap is None:
                # L'entorn oficial no informa la capacitat: es dedueix de soc_kwh / soc_pct i
                # s'arrodoneix (l'error de coma flotant donaria p_max = 1500.0000000000002).
                pct = float(obs.get("soc_pct", 0.0) or 0.0)
                cap = round(soc_kwh / pct) if (pct > 1e-9 and soc_kwh > 1e-6) else self.cap
                if not (100.0 <= cap <= 1e6):
                    cap = self.cap
            self.cap = float(cap)
            self.pmax = float(obs.get("power_max_kw", self.cap / 4.0))
        except Exception:
            soc_kwh = float(obs.get("soc_kwh", 0.0))
        try:
            if not self.ready:
                raise RuntimeError("sense artefacte")
            t = hour_index(obs["timestamp"])
            if t < 200 or t + HORIZON + 2 >= N_HOURS:
                raise RuntimeError("timestamp fora de rang")
            prices, co2s = self._update_history(t, obs)
            self.prev_idx = t
            a = self.decide(t, soc_kwh, prices, co2s)
            if a is None or a != a:
                raise RuntimeError("LP sense solucio")
            consigna_desitjada = a
        except Exception:
            self.n_fallback += 1
            try:
                consigna_desitjada = self._heuristic(obs)
            except Exception:
                consigna_desitjada = 0.0

        # Clamp fisic exacte (zero infraccions)
        cap_max = float(self.cap)
        p_max = float(self.pmax)
        eta = 0.90 ** 0.5
        eps = 1e-4
        p_max = p_max - 1e-6                      # marge numeric sobre el limit de l'inversor
        max_carrega = max(0.0, min(p_max, (cap_max - soc_kwh) / eta - eps))
        max_descarrega = max(0.0, min(p_max, soc_kwh * eta - eps))
        consigna_segura = float(max(-max_descarrega, min(max_carrega, consigna_desitjada)))
        if consigna_segura != consigna_segura:
            consigna_segura = 0.0
        return consigna_segura

In [ ]:
%%writefile hm_build/data_prep.py
"""
src/data_prep.py
Preprocessat offline: CSV de 5 minuts de data/train/ -> taula horaria neta.
Usat per train_pipeline.ipynb i run_hard_sim.py (NO forma part de submission.zip).
"""
from pathlib import Path

import numpy as np
import pandas as pd

def find_train_dir():
    """Localitza data/train (repo local o dataset de Kaggle a /kaggle/input/**)."""
    here = Path(__file__).resolve().parent
    cands = [here.parent / "data" / "train", here / "data" / "train", Path("data/train"),
             Path("../data/train"), Path("../../data/train")]
    for c in cands:
        if (c / "contador.csv").exists():
            return c.resolve()
    kag = Path("/kaggle/input")
    if kag.exists():
        for f in kag.rglob("contador.csv"):
            return f.parent
    return cands[0]


TRAIN_DIR = find_train_dir()
PV_SCALE = 12.0          # Mode Dificil: solar x12
MAX_LOAD_KW = 5000.0     # tall fisic recomanat a GUIA_VARIABLES.md

COL_LOAD = "Contador - POTENCIA ACTIVA EXPORTADA (kW)"
METEO_COLS = {
    "Atersa - IRRADIANCIA (W/m2)": "irradiancia_wm2",
    "Atersa - TEMPERATURA AMBIENTE (SENSOR 1) (ºC)": "temp_ambient_c",
    "Atersa - TEMPERATURA PANEL (SENSOR 1) (ºC)": "temp_panel_c",
    "Atersa - VELOCIDAD VIENTO (SENSOR 1) (m/s)": "vent_ms",
}


def _hourly_mean(df):
    # Mateixa convencio que starter_kit/tutorial.ipynb (i per tant que el test ocult).
    return df.resample("1h").mean()


def build_hourly(train_dir=None, pv_scale=PV_SCALE):
    """Taula horaria: load_kw (0.0/aberrants -> NaN), pv_kw (x pv_scale), meteo, preu, CO2.
    No s'imputa res aqui: els forats (novembre 2025, ultims 7 dies de cada mes) queden com NaN."""
    train_dir = Path(train_dir or TRAIN_DIR)
    rd = lambda n, cols: pd.read_csv(train_dir / n, usecols=["Fecha", *cols],
                                     parse_dates=["Fecha"]).set_index("Fecha").sort_index()
    cont = rd("contador.csv", [COL_LOAD])
    load = -cont[COL_LOAD]
    load = load.where((load > 0.0) & (load < MAX_LOAD_KW))      # 0.0 = caiguda de sensor -> NaN
    load_h = _hourly_mean(load.to_frame("load_kw"))

    inv_cols = [c for c in pd.read_csv(train_dir / "inversor.csv", nrows=0).columns
                if c.endswith("- POTENCIA ACTIVA (kW)")]
    inv = rd("inversor.csv", inv_cols)
    pv = inv.clip(lower=0).sum(axis=1, min_count=1) * pv_scale
    pv_h = _hourly_mean(pv.to_frame("pv_kw"))

    met = rd("meteo.csv", list(METEO_COLS)).rename(columns=METEO_COLS)
    met_h = _hourly_mean(met)

    price = rd("precio_pvpc.csv", ["precio_eur_per_kwh"])
    co2 = rd("co2_red.csv", ["co2_kg_per_kwh"])
    df = pd.concat([price, co2, load_h, pv_h, met_h], axis=1, sort=True)
    df = df[df["precio_eur_per_kwh"].notna()]
    df.index.name = "Fecha"
    return df


def build_env_frame(train_dir=None, pv_scale=PV_SCALE):
    """Taula 'oficial' per a CampusBatteryEnv, calcada de starter_kit/tutorial.ipynb
    (load clip [0, 5000] amb els 0.0 conservats, PV fillna(0), dropna) + PV x pv_scale
    + meteo horaria (per a noves_dades_5min en Mode Dificil)."""
    train_dir = Path(train_dir or TRAIN_DIR)
    cont = pd.read_csv(train_dir / "contador.csv", usecols=["Fecha", COL_LOAD],
                       parse_dates=["Fecha"]).set_index("Fecha")
    load = (-cont[COL_LOAD]).clip(lower=0, upper=5000).resample("1h").mean().dropna()
    inv_cols = [c for c in pd.read_csv(train_dir / "inversor.csv", nrows=0).columns
                if c.endswith("- POTENCIA ACTIVA (kW)")]
    inv = pd.read_csv(train_dir / "inversor.csv", usecols=["Fecha", *inv_cols],
                      parse_dates=["Fecha"]).set_index("Fecha")
    pv = inv.fillna(0).sum(axis=1).clip(lower=0).resample("1h").mean().dropna() * pv_scale
    met = pd.read_csv(train_dir / "meteo.csv", usecols=["Fecha", *METEO_COLS],
                      parse_dates=["Fecha"]).set_index("Fecha").rename(columns=METEO_COLS)
    met = met.resample("1h").mean()
    price = pd.read_csv(train_dir / "precio_pvpc.csv", parse_dates=["Fecha"]).set_index("Fecha")
    co2 = pd.read_csv(train_dir / "co2_red.csv", parse_dates=["Fecha"]).set_index("Fecha")
    df = pd.DataFrame({"load_kw": load, "pv_kw": pv,
                       "precio_eur_per_kwh": price["precio_eur_per_kwh"],
                       "co2_kg_per_kwh": co2["co2_kg_per_kwh"]}).dropna()
    df = df.join(met)
    df.index.name = "Fecha"
    return df

In [ ]:
%%writefile hm_build/hard_env.py
"""
src/hard_env.py
Simulador local del MODE DIFICIL. Reutilitza al 100% la fisica i la metrica de
`starter_kit.environment.CampusBatteryEnv` (no es modifica) i nomes ajusta l'observacio:

  * preus_properes_24h / co2_properes_24h retallats a spec["forecast_horizon"] (3 h)
  * obs["cap_max_kwh"], obs["power_max_kw"]
  * meteo de la darrera hora (fila t-1) dins obs["noves_dades_5min"]

Us:
    env = HardModeEnv(df_hard, spec=SPEC_HARD)     # df_hard["pv_kw"] ja escalat x12
"""
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    from starter_kit.environment import CampusBatteryEnv  # noqa: E402
except Exception:                       # Kaggle: environment.py al costat d'aquest fitxer
    from environment import CampusBatteryEnv  # noqa: E402

SPEC_HARD = {
    "capacity_kwh": 6000.0,
    "power_kw": 1500.0,
    "efficiency": 0.90,
    "soc_init": 0.50,
    "soc_min": 0.0,
    "soc_max": 1.0,
    "dt_hours": 1.0,
    "forecast_horizon": 3,
}
METEO_KEYS = ("irradiancia_wm2", "temp_ambient_c", "temp_panel_c", "vent_ms")


class HardModeEnv(CampusBatteryEnv):
    def __init__(self, hourly_data, spec=None, export_price=0.0):
        super().__init__(hourly_data, spec=dict(spec or SPEC_HARD), export_price=export_price)

    def _get_obs(self):
        obs = super()._get_obs()
        if obs is None:
            return None
        h = int(self.spec.get("forecast_horizon", 24))
        obs["preus_properes_24h"] = obs["preus_properes_24h"][:h]
        obs["co2_properes_24h"] = obs["co2_properes_24h"][:h]
        obs["cap_max_kwh"] = self.cap
        obs["power_max_kw"] = self.power
        row = self.df.iloc[self.t - 1] if self.t > 0 else self.df.iloc[self.t]
        for k in METEO_KEYS:
            if k in self.df.columns:
                obs["noves_dades_5min"][k] = float(row[k])
        return obs


def run_hard(agent, df_hard, spec=None, return_env=False, desc=None):
    """Simulacio sequencial completa en Mode Dificil. Retorna el diccionari de metriques.
    `desc`: si es dona (i hi ha tqdm), mostra una barra de progres dels passos."""
    import time
    env = HardModeEnv(df_hard, spec=spec)
    obs = env.reset()
    done = False
    lat = []
    bar = None
    if desc:
        try:
            from tqdm.auto import tqdm
            bar = tqdm(total=env.n, desc=desc, leave=False, mininterval=0.5)
        except Exception:
            bar = None
    while not done:
        t0 = time.perf_counter()
        a = agent.act(obs)
        lat.append(time.perf_counter() - t0)
        obs, done = env.step(a)
        if bar is not None:
            bar.update(1)
    if bar is not None:
        bar.close()
    res = env.evaluate_results()
    res["ms_per_step_mean"] = round(1000 * sum(lat) / len(lat), 3)
    res["ms_per_step_max"] = round(1000 * max(lat), 3)
    return (res, env) if return_env else res

In [ ]:
%%writefile hm_build/train_lib.py
"""
src/train_lib.py
Funcions d'entrenament / validacio offline per al Mode Dificil.
Les fa servir train_pipeline.ipynb. NO forma part de submission.zip.
"""
import time
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import linprog
from scipy.sparse import lil_matrix

try:
    from tqdm.auto import tqdm
except Exception:                       # sense tqdm: iterador normal
    def tqdm(it=None, **kw):
        return it

import agent as A
from data_prep import PV_SCALE
from hard_env import HardModeEnv, SPEC_HARD, run_hard

COLMAP = {"load": "load_kw", "pv": "pv_kw", "irr": "irradiancia_wm2", "tamb": "temp_ambient_c",
          "tpan": "temp_panel_c", "price": "precio_eur_per_kwh", "co2": "co2_kg_per_kwh"}
TARGETS = ("load", "pv", "price", "co2")
K_MIN = {"load": 0, "pv": 0, "price": 3, "co2": 3}     # preu/CO2: k=0..2 son exactes


# ----------------------------------------------------------------------------------
# Dades -> arrays sobre l'eix absolut de l'agent
# ----------------------------------------------------------------------------------
def to_arrays(hourly: pd.DataFrame) -> dict:
    """Taula horaria neta -> dict d'arrays (N_HOURS,) amb NaN on no hi ha dada.
    Imputacio de forats curts (<= 3 h) per interpolacio lineal; els forats llargs
    (novembre 2025, setmanes de test) NO s'interpolen ni es propaguen."""
    idx = A.full_index()
    out = {}
    for name, col in COLMAP.items():
        s = hourly[col].reindex(idx)
        if name in A.TELEMETRY:
            s = interpolate_short(s, 3)
        out[name] = s.values.astype(float)
    return out


def interpolate_short(s: pd.Series, max_gap: int = 3) -> pd.Series:
    """Interpola nomes forats interiors de longitud <= max_gap hores."""
    isna = s.isna()
    grp = (isna != isna.shift()).cumsum()
    run = isna.groupby(grp).transform("size")
    filled = s.interpolate(limit_area="inside")
    return s.where(~(isna & (run <= max_gap)), filled)


def impute_profile(s: pd.Series) -> pd.Series:
    """Estrategia alternativa (per comparar): mediana mes / dia-setmana / hora."""
    key = [s.index.month, s.index.dayofweek, s.index.hour]
    med = s.groupby(key).transform("median")
    return s.fillna(med)


def clear_sky_table(pv: np.ndarray, q: float = 0.97, win: int = 15) -> np.ndarray:
    """Envolupant de cel clar (367, 24): quantil alt de pv per (dia de l'any +-win, hora)."""
    idx = A.full_index()
    df = pd.DataFrame({"pv": pv, "doy": idx.dayofyear, "h": idx.hour}).dropna()
    tab = np.full((367, 24), np.nan)
    for h in range(24):
        d = df[df.h == h]
        if d.empty:
            continue
        doy = d.doy.values
        v = d.pv.values
        for day in range(1, 367):
            dist = np.abs(doy - day)
            dist = np.minimum(dist, 366 - dist)
            m = dist <= win
            if m.sum() >= 5:
                tab[day, h] = np.quantile(v[m], q)
    # forats (novembre): interpolacio circular al llarg de l'any
    for h in range(24):
        col = pd.Series(np.concatenate([tab[1:, h]] * 3))
        col = col.interpolate(limit_direction="both")
        tab[1:, h] = col.values[366:732]
    tab[0] = tab[1]
    tab = np.nan_to_num(tab, nan=0.0)
    tab[tab < 5.0] = 0.0
    return tab


# ----------------------------------------------------------------------------------
# Blocs de validacio (mirall del test ocult)
# ----------------------------------------------------------------------------------
def validation_blocks(env_df: pd.DataFrame):
    """Per cada mes de train: els darrers 168 h consecutius disponibles (dies ~17-23),
    es a dir, la setmana immediatament anterior a la setmana oculta."""
    blocks = []
    for per, g in env_df.groupby(env_df.index.to_period("M")):
        if len(g) < 336:
            continue
        end = g.index.max()
        start = end - pd.Timedelta(hours=167)
        blk = env_df.loc[start:end]
        if len(blk) >= 150:
            blocks.append((start, end))
    return blocks


def synthetic_november(env_df: pd.DataFrame, hourly: pd.DataFrame) -> pd.DataFrame:
    """Bloc sintetic 17-23 Nov 2025: preu i CO2 REALS de novembre + consum / solar / meteo
    = mitjana de dues setmanes reals alineades per dia de la setmana (13-19 Oct i 1-7 Des)."""
    idx = pd.date_range("2025-11-17", periods=168, freq="1h")
    a = env_df.loc[NOV_SOURCES[0][0]:NOV_SOURCES[0][1]]
    b = env_df.loc[NOV_SOURCES[1][0]:NOV_SOURCES[1][1]]
    cols = ["load_kw", "pv_kw", "temp_ambient_c", "temp_panel_c", "vent_ms", "irradiancia_wm2"]
    a = a.reindex(pd.date_range("2025-10-13", periods=168, freq="1h"))[cols].interpolate(limit_direction="both")
    b = b.reindex(pd.date_range("2025-12-01", periods=168, freq="1h"))[cols].interpolate(limit_direction="both")
    out = pd.DataFrame(0.5 * (a.values + b.values), index=idx, columns=cols)
    out["precio_eur_per_kwh"] = hourly["precio_eur_per_kwh"].reindex(idx).values
    out["co2_kg_per_kwh"] = hourly["co2_kg_per_kwh"].reindex(idx).values
    out.index.name = "Fecha"
    return out[env_df.columns]


def make_validation_frame(env_df, hourly):
    """12 blocs de 168 h concatenats (11 reals + novembre sintetic), ordre cronologic."""
    blocks = validation_blocks(env_df)
    parts = [env_df.loc[s:e] for s, e in blocks]
    nov = synthetic_november(env_df, hourly)
    parts.append(nov)
    blocks.append((nov.index.min(), nov.index.max()))
    order = np.argsort([p.index.min() for p in parts])
    parts = [parts[i] for i in order]
    blocks = [blocks[i] for i in order]
    return pd.concat(parts), blocks


# Setmanes reals amb que es construeix el bloc sintetic de novembre: tambe s'han d'amagar
# de l'entrenament i de l'historial de l'agent, o la validacio de novembre seria optimista.
NOV_SOURCES = [(pd.Timestamp("2025-10-13"), pd.Timestamp("2025-10-19 23:00")),
               (pd.Timestamp("2025-12-01"), pd.Timestamp("2025-12-07 23:00"))]


def mask_blocks(arrays: dict, blocks) -> dict:
    """Copia dels arrays amb els blocs de validacio amagats (NaN). Si hi ha el bloc sintetic
    de novembre, tambe s'amaga la telemetria de les seves dues setmanes font."""
    out = {k: v.copy() for k, v in arrays.items()}
    if any(s.month == 11 for s, _ in blocks):
        for s, e in NOV_SOURCES:
            i0, i1 = A.hour_index(s), A.hour_index(e) + 1
            for k in A.TELEMETRY:
                out[k][i0:i1] = np.nan
    for s, e in blocks:
        i0, i1 = A.hour_index(s), A.hour_index(e) + 1
        for k in out:
            out[k][i0:i1] = np.nan
    return out


# ----------------------------------------------------------------------------------
# Datasets supervisats (mateix codi de features que l'agent)
# ----------------------------------------------------------------------------------
def nov_gap_mask() -> np.ndarray:
    """Hores t la finestra de lags de les quals (fins a 168 h) creua el forat Oct->Des 2025."""
    m = np.zeros(A.N_HOURS, dtype=bool)
    m[A.hour_index("2025-11-01"):A.hour_index("2025-12-08")] = True
    return m


def build_dataset(arrays: dict, cs_table: np.ndarray, target: str, t_range=None):
    """Retorna X (n, F), y (n,), t (n,), k (n,) per a totes les parelles (t, k) amb
    objectiu conegut. Les features es construeixen amb agent.Features (identic a online)."""
    idx = A.full_index()
    cal = A.calendar_matrix(idx)
    cs = A.clear_sky_series(cs_table, idx)
    feat = A.Features(arrays, cal, cs)
    y_all = arrays[target]
    anchor = arrays["price"]        # hores amb mercat conegut = hores de train
    gap = nov_gap_mask()
    if t_range is None:
        t_range = range(200, A.N_HOURS - A.HORIZON - 2)
    Xs, ys, ts, ks = [], [], [], []
    kmin = K_MIN[target]
    for t in t_range:
        if anchor[t] != anchor[t]:
            continue
        if target in ("load", "pv"):
            if gap[t]:
                continue                       # no entrenem a traves del forat de novembre
            if y_all[t - 1] != y_all[t - 1]:
                continue                       # sense telemetria recent (inici de segment)
        else:
            if anchor[t + 2] != anchor[t + 2] or anchor[t - 1] != anchor[t - 1]:
                continue
        y = y_all[t:t + A.HORIZON]
        ok = (y == y)
        ok[:kmin] = False
        if not ok.any():
            continue
        X = feat.build(target, t)
        Xs.append(X[ok]); ys.append(y[ok])
        ts.append(np.full(int(ok.sum()), t)); ks.append(A._K[ok])
    return (np.concatenate(Xs), np.concatenate(ys), np.concatenate(ts), np.concatenate(ks))


# ----------------------------------------------------------------------------------
# Families de models
# ----------------------------------------------------------------------------------
def gpu_available() -> bool:
    try:
        import subprocess
        return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except Exception:
        return False


LGB_PARAMS = dict(objective="l1", learning_rate=0.1, num_leaves=31, min_data_in_leaf=40,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                  verbose=-1, seed=7, num_threads=4)


def fit_lgb(X, y, n_rounds=400, params=None, names=None):
    import lightgbm as lgb
    p = dict(LGB_PARAMS)
    p.update(params or {})
    ds = lgb.Dataset(X, y, feature_name=list(names) if names else "auto")
    return lgb.train(p, ds, num_boost_round=n_rounds)


def fit_xgb(X, y, n_rounds=400, use_gpu=False):
    import xgboost as xgb
    m = xgb.XGBRegressor(n_estimators=n_rounds, learning_rate=0.05, max_depth=7, subsample=0.8,
                         colsample_bytree=0.8, objective="reg:absoluteerror", tree_method="hist",
                         device="cuda" if use_gpu else "cpu", n_jobs=4, random_state=7)
    return m.fit(X, y)


def fit_catboost(X, y, n_rounds=400, use_gpu=False):
    from catboost import CatBoostRegressor
    m = CatBoostRegressor(iterations=n_rounds, learning_rate=0.08, depth=7, loss_function="MAE",
                          task_type="GPU" if use_gpu else "CPU", verbose=0, random_seed=7,
                          metric_period=n_rounds, allow_writing_files=False,   # sense avisos GPU
                          thread_count=4)
    return m.fit(X, y)


def fit_rf(X, y):
    from sklearn.ensemble import RandomForestRegressor
    Xf = np.where(np.isnan(X), -999.0, X)
    m = RandomForestRegressor(n_estimators=60, min_samples_leaf=5, max_features=0.5, n_jobs=4,
                              random_state=7).fit(Xf, y)
    m._nanfill = True
    return m


def fit_ridge(X, y, k, alpha=10.0):
    """Ridge directe: un vector de pesos per horitzo k. Retorna l'especificacio numpy
    {W (24,F), b (24,), mu, sd} que l'agent pot avaluar sense dependencies."""
    from sklearn.linear_model import Ridge
    mu = np.nanmean(X, axis=0)
    mu = np.where(np.isnan(mu), 0.0, mu)
    Xi = np.where(np.isnan(X), mu, X)
    sd = Xi.std(axis=0)
    sd = np.where(sd < 1e-9, 1.0, sd)
    Z = (Xi - mu) / sd
    F = X.shape[1]
    W = np.zeros((A.HORIZON, F)); b = np.zeros(A.HORIZON)
    for kk in range(A.HORIZON):
        m = k == kk
        if m.sum() < 50:
            continue
        r = Ridge(alpha=alpha).fit(Z[m], y[m])
        W[kk] = r.coef_; b[kk] = r.intercept_
    return {"W": W, "b": b, "mu": mu, "sd": sd}


def predict_ridge(spec, X, k):
    Z = (np.where(np.isnan(X), spec["mu"], X) - spec["mu"]) / spec["sd"]
    return (Z * spec["W"][k]).sum(axis=1) + spec["b"][k]


def predict_any(model, X, k):
    if isinstance(model, dict):
        return predict_ridge(model, X, k)
    if getattr(model, "_nanfill", False):
        return model.predict(np.where(np.isnan(X), -999.0, X))
    return model.predict(X)


def seasonal_naive(X, names):
    """Baseline: valor de la mateixa hora d'ahir (lag 24 h de l'hora objectiu)."""
    col = [n for n in names if n.endswith("_h24")][0]
    j = list(names).index(col)
    fb = [n for n in names if n in ("l1", "pv1", "y2")][0]
    p = X[:, j].copy()
    p[np.isnan(p)] = X[np.isnan(p), list(names).index(fb)]
    return p


def mae(y, p):
    m = ~np.isnan(p) & ~np.isnan(y)
    return float(np.mean(np.abs(y[m] - p[m])))


def benchmark_target(target, arrays_tr, arrays_full, cs_table, blocks, families, use_gpu=False,
                     n_rounds=400):
    """Entrena cada familia sense els blocs de validacio i mesura MAE sobre els blocs.
    Les features de validacio fan servir l'historial real (com passara al test)."""
    names = A.FEATURE_NAMES[target]
    Xtr, ytr, ttr, ktr = build_dataset(arrays_tr, cs_table, target)
    t_val = np.concatenate([np.arange(A.hour_index(s), A.hour_index(e) + 1) for s, e in blocks])
    Xva, yva, tva, kva = build_dataset(arrays_full, cs_table, target, t_range=t_val)
    in_blk = np.isin(tva + kva, t_val)
    Xva, yva, kva = Xva[in_blk], yva[in_blk], kva[in_blk]
    rows, models = [], {}
    bar = tqdm(list(families), desc=f"models [{target}]", leave=False)
    for fam in bar:
        if hasattr(bar, "set_postfix_str"):
            bar.set_postfix_str(fam)
        t0 = time.time()
        if fam == "seasonal_naive":
            model = None; p = seasonal_naive(Xva, names)
        elif fam == "ridge":
            model = fit_ridge(Xtr, ytr, ktr); p = predict_any(model, Xva, kva)
        elif fam == "lightgbm_stacked":
            model = fit_lgb(Xtr, ytr, n_rounds, names=names); p = model.predict(Xva)
        elif fam == "lightgbm_buckets":
            model = fit_lgb_buckets(Xtr, ytr, ktr, target); p = predict_buckets(model, Xva, kva)
            rdg = fit_ridge(Xtr, ytr, ktr)             # k >= 12: Ridge (com fa l'agent)
            p = np.where(np.isnan(p), predict_ridge(rdg, Xva, kva), p)
        elif fam == "xgboost":
            model = fit_xgb(Xtr, ytr, n_rounds, use_gpu); p = model.predict(Xva)
        elif fam == "catboost":
            model = fit_catboost(Xtr, ytr, n_rounds, use_gpu); p = model.predict(Xva)
        elif fam == "random_forest":
            model = fit_rf(Xtr, ytr); p = predict_any(model, Xva, kva)
        else:
            raise ValueError(fam)
        short = kva <= 5
        rows.append({"target": target, "model": fam, "MAE": mae(yva, p),
                     "MAE_k<=5": mae(yva[short], p[short]), "MAE_k>5": mae(yva[~short], p[~short]),
                     "n_train": len(ytr), "n_val": len(yva), "fit_s": round(time.time() - t0, 1)})
        models[fam] = model
    return pd.DataFrame(rows), models


LGB_OBJECTIVE = {"load": "l2", "pv": "l1", "price": "l2", "co2": "l1"}


def fit_lgb_buckets(X, y, k, target, n_rounds=150, params=None):
    """LightGBM per grup d'horitzo (agent.BUCKETS). Retorna [(k0, k1, booster), ...]."""
    out = []
    p = {"objective": LGB_OBJECTIVE[target]}
    p.update(params or {})
    for k0, k1 in A.BUCKETS:
        m = (k >= k0) & (k <= k1)
        if m.sum() < 200:
            continue
        out.append((k0, k1, fit_lgb(X[m], y[m], n_rounds, params=p, names=A.FEATURE_NAMES[target])))
    return out


def predict_buckets(models, X, k):
    p = np.full(len(k), np.nan)
    for k0, k1, bst in models:
        m = (k >= k0) & (k <= k1)
        if m.any():
            p[m] = bst.predict(X[m])
    return p


def train_final_models(arrays, cs_table, n_rounds=150, lgb_params=None, cap_qs=(0.2,)):
    # lgb_params: None, o dict per objectiu {"load": {"n_rounds": 200, "num_leaves": 31, ...}}
    """Entrena LightGBM per grups d'horitzo + Ridge (reserva) per als 4 objectius. Retorna
    l'especificacio serialitzable de l'agent (cadenes de model LightGBM + pesos numpy)."""
    out = {}
    for target in tqdm(TARGETS, desc="entrenament LightGBM+Ridge", leave=False):
        X, y, t, k = build_dataset(arrays, cs_table, target)
        lp = dict((lgb_params or {}).get(target, {}))
        nr = int(lp.pop("n_rounds", n_rounds))
        bk = fit_lgb_buckets(X, y, k, target, nr, lp or None)
        out[target] = {"lgb": [(k0, k1, b.model_to_string()) for k0, k1, b in bk],
                       "ridge": fit_ridge(X, y, k), "features": list(A.FEATURE_NAMES[target])}
    out["cap"] = train_cap_models(arrays, cs_table, cap_qs)
    return out


def train_cap_models(arrays, cs_table, qs, n_rounds=150):
    """Models quantil a k=0: consum al quantil q i solar al quantil 1-q. La seva diferencia
    es una cota inferior prudent del consum net -> sostre de descarrega sense exportacio."""
    out = {}
    if not len(qs):
        return out
    Xl, yl, _, kl = build_dataset(arrays, cs_table, "load")
    Xp, yp, _, kp = build_dataset(arrays, cs_table, "pv")
    Xl, yl, Xp, yp = Xl[kl == 0], yl[kl == 0], Xp[kp == 0], yp[kp == 0]
    for q in qs:
        bl = fit_lgb(Xl, yl, n_rounds, params={"objective": "quantile", "alpha": float(q)},
                     names=A.FEATURE_NAMES["load"])
        bp = fit_lgb(Xp, yp, n_rounds, params={"objective": "quantile", "alpha": 1.0 - float(q)},
                     names=A.FEATURE_NAMES["pv"])
        out[float(q)] = (bl.model_to_string(), bp.model_to_string())
    return out


# ----------------------------------------------------------------------------------
# Optuna: hiperparametres dels models i del control
# ----------------------------------------------------------------------------------
def optuna_tune_lgb(target, arrays_tr, arrays_full, cs_table, blocks, n_trials=20, seed=7):
    """Cerca bayesiana (TPE) dels hiperparametres de LightGBM per a un objectiu, minimitzant
    el MAE fora de mostra sobre els blocs de validacio (horitzons amb arbres, k <= 11).
    Retorna (millors_parametres, study). El primer assaig son els valors per defecte."""
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    Xtr, ytr, _, ktr = build_dataset(arrays_tr, cs_table, target)
    t_val = np.concatenate([np.arange(A.hour_index(s), A.hour_index(e) + 1) for s, e in blocks])
    Xva, yva, tva, kva = build_dataset(arrays_full, cs_table, target, t_range=t_val)
    ok = np.isin(tva + kva, t_val) & (kva <= A.BUCKETS[-1][1])
    Xva, yva, kva = Xva[ok], yva[ok], kva[ok]

    def objective(trial):
        p = {"learning_rate": trial.suggest_float("learning_rate", 0.03, 0.2, log=True),
             "num_leaves": trial.suggest_int("num_leaves", 15, 63),
             "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 200),
             "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
             "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
             "lambda_l2": trial.suggest_float("lambda_l2", 0.1, 10.0, log=True)}
        nr = trial.suggest_int("n_rounds", 80, 300)      # limitat per la latencia de l'agent
        bk = fit_lgb_buckets(Xtr, ytr, ktr, target, nr, p)
        return mae(yva, predict_buckets(bk, Xva, kva))

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.enqueue_trial({"learning_rate": LGB_PARAMS["learning_rate"], "num_leaves": LGB_PARAMS["num_leaves"],
                         "min_data_in_leaf": LGB_PARAMS["min_data_in_leaf"],
                         "feature_fraction": LGB_PARAMS["feature_fraction"],
                         "bagging_fraction": LGB_PARAMS["bagging_fraction"],
                         "lambda_l2": LGB_PARAMS["lambda_l2"], "n_rounds": 150})
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return dict(study.best_params), study


CONTROL_SPACE = {
    "sigma_scale": ("float", 0.0, 1.5), "cap_q": ("cat", [0.0, 0.1, 0.2, 0.3]),
    "buf_kw": ("float", 0.0, 40.0), "gamma": ("float", 0.8, 1.1), "w_co2": ("float", 0.6, 0.8),
    "net_bias_kw": ("float", -40.0, 40.0), "blend_load": ("float", 0.2, 0.8),
    "blend_pv": ("float", 0.2, 0.8), "blend_price": ("float", 0.2, 0.8),
    "blend_co2": ("float", 0.2, 0.8),
}


def _control_params(flat):
    """Parametres plans d'Optuna -> parametres de l'agent."""
    p = {k: v for k, v in flat.items() if not k.startswith("blend_")}
    p["blend_w"] = {t: float(flat.get("blend_" + t, 0.5)) for t in TARGETS}
    return p


def split_scores(env, blocks):
    """Puntuacio oficial sobre els blocs parells (A) i senars (B) per separat."""
    ts = pd.DatetimeIndex(env.df["Fecha"])
    out = []
    for half in (0, 1):
        m = np.zeros(len(ts), dtype=bool)
        for i, (s, e) in enumerate(blocks):
            if i % 2 == half:
                m |= np.asarray((ts >= s) & (ts <= e))
        out.append(env.evaluate_results(mask=m)["score"])
    return out


def optuna_tune_control(artifact, val_df, blocks, n_trials=60, base=None, seed=7):
    """Cerca bayesiana (TPE) dels parametres del control contra la metrica oficial en Mode
    Dificil (aproximacio parametrica de la funcio de cost, ajustada sobre el simulador).

    Antisobreajust: nomes s'accepta una configuracio si millora la de per defecte a les DUES
    meitats dels blocs (parells i senars) i el guany total supera `min_gain`.
    Retorna (parametres_escollits, study, taula)."""
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    base = dict(base or {})
    D0 = A.DEFAULT_PARAMS
    default_flat = {"sigma_scale": D0["sigma_scale"], "cap_q": D0["cap_q"], "buf_kw": D0["buf_kw"],
                    "gamma": D0["gamma"], "w_co2": D0["w_co2"], "net_bias_kw": D0["net_bias_kw"],
                    "blend_load": 0.5, "blend_pv": 0.5, "blend_price": 0.5, "blend_co2": 0.5}
    caps = set((artifact["models"].get("cap") or {}).keys())

    def objective(trial):
        flat = {}
        for name, spec in CONTROL_SPACE.items():
            if spec[0] == "cat":
                choices = [c for c in spec[1] if c == 0.0 or c in caps] if name == "cap_q" else spec[1]
                flat[name] = trial.suggest_categorical(name, choices)
            else:
                flat[name] = trial.suggest_float(name, spec[1], spec[2])
        res, env = simulate(artifact, val_df, {**base, **_control_params(flat)}, return_env=True)
        a, b = split_scores(env, blocks)
        trial.set_user_attr("score_A", a); trial.set_user_attr("score_B", b)
        trial.set_user_attr("infraccions", res["infraccions"])
        trial.set_user_attr("ms", res["ms_per_step_mean"])
        return res["score"] - 1000.0 * (res["infraccions"] > 0)

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.enqueue_trial(default_flat)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    rows = [{"trial": t.number, "score": t.value, "score_A": t.user_attrs.get("score_A"),
             "score_B": t.user_attrs.get("score_B"), "ms/pas": t.user_attrs.get("ms"), **t.params}
            for t in study.trials if t.value is not None]
    tab = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    d = tab[tab.trial == 0].iloc[0]
    min_gain = 0.05
    okm = (tab.score_A > d.score_A) & (tab.score_B > d.score_B) & (tab.score >= d.score + min_gain)
    chosen = tab[okm].iloc[0] if okm.any() else d
    flat = {k: (float(chosen[k]) if k != "cap_q" else float(chosen[k])) for k in CONTROL_SPACE}
    tab.attrs["chosen_trial"] = int(chosen.trial)
    return {**base, **_control_params(flat)}, study, tab


# ----------------------------------------------------------------------------------
# Artefacte per a l'agent
# ----------------------------------------------------------------------------------
def baseline_refs(env_df: pd.DataFrame):
    """Mitjanes ponderades pel baseline (sense bateria): normalitzen l'objectiu 70/30."""
    base = np.maximum(env_df["load_kw"] - env_df["pv_kw"], 0.0)
    return (float((base * env_df["co2_kg_per_kwh"]).sum() / base.sum()),
            float((base * env_df["precio_eur_per_kwh"]).sum() / base.sum()))


def residual_sigma(models, arrays_full, cs_table, blocks, prefer="blend"):
    """Desviacio (RMSE) de l'error de prediccio per horitzo k, mesurada fora de mostra sobre
    els blocs de validacio amb models que no els han vist: consum en kW i solar com a
    fraccio de l'envolupant de cel clar. Alimenta el DP estocastic de l'agent."""
    idx = A.full_index()
    cs = A.clear_sky_series(cs_table, idx)
    out = {}
    t_val = np.concatenate([np.arange(A.hour_index(s), A.hour_index(e) + 1) for s, e in blocks])
    for tg, key in (("load", "load"), ("pv", "pv_frac")):
        X, y, t, k = build_dataset(arrays_full, cs_table, tg, t_range=t_val)
        ok = np.isin(t + k, t_val)
        X, y, t, k = X[ok], y[ok], t[ok], k[ok]
        # mateixa combinacio que agent.Forecaster, vectoritzada per grup d'horitzo
        import lightgbm as lgb
        p = np.full(len(y), np.nan)
        if prefer != "ridge":
            for k0, k1, txt in models[tg]["lgb"]:
                m = (k >= k0) & (k <= k1)
                if m.any():
                    p[m] = lgb.Booster(model_str=txt).predict(X[m])
        r = predict_ridge(models[tg]["ridge"], X, k)
        bw = 0.5 if prefer == "blend" else 0.0
        p = np.where(np.isnan(p), r, (1.0 - bw) * p + bw * r)
        err = y - p
        sig = np.zeros(A.HORIZON)
        for kk in range(A.HORIZON):
            m = (k == kk) & ~np.isnan(err)
            if tg == "pv":
                c = cs[t[m] + kk]
                day = c > 50.0
                sig[kk] = float(np.sqrt(np.mean((err[m][day] / c[day]) ** 2))) if day.any() else 0.0
            else:
                sig[kk] = float(np.sqrt(np.mean(err[m] ** 2))) if m.any() else 0.0
        out[key] = sig
    return out


def make_artifact(arrays, cs_table, models, env_df, params=None, sigma=None):
    co2_ref, price_ref = baseline_refs(env_df)
    last = max(np.flatnonzero(~np.isnan(v)).max() for v in arrays.values()) + 1
    m = (~np.isnan(arrays["pv"])) & (arrays["pv"] > 50) & ~np.isnan(arrays["irr"])
    irr_per_pv = float(np.median(arrays["irr"][m] / arrays["pv"][m])) if m.any() else 0.6
    p = {"co2_ref": co2_ref, "price_ref": price_ref}
    p.update(params or {})
    return {"hist": {k: v[:last].astype(np.float32) for k, v in arrays.items()},
            "cs_table": cs_table.astype(np.float32), "models": models,
            "irr_per_pv": irr_per_pv, "params": p, "pv_scale": PV_SCALE,
            "sigma": {k: np.asarray(v, dtype=float) for k, v in (sigma or {}).items()}}


# ----------------------------------------------------------------------------------
# Simulacio i cotes
# ----------------------------------------------------------------------------------
def simulate(artifact, df_env, params=None, spec=None, return_env=False, desc=None):
    art = dict(artifact)
    art["params"] = dict(artifact.get("params", {}))
    art["params"].update(params or {})
    agent = A.BatteryAgent(artifact=art)
    assert agent.ready, "l'agent no ha pogut carregar l'artefacte"
    out = run_hard(agent, df_env, spec=spec or SPEC_HARD, return_env=return_env, desc=desc)
    (out[0] if return_env else out)["fallbacks"] = agent.n_fallback
    return out


class ModelAgent(A.BatteryAgent):
    """Mateix agent i mateix control, pero amb les prediccions de qualsevol familia de
    models (per comparar families amb la metrica oficial; no es limita la latencia)."""

    def set_family(self, family, models):
        self.family = family
        self.fam_models = models        # dict objectiu -> model

    def _predict_all(self, t, Xl, Xp):
        f = self.feat
        Xs = {"load": Xl, "pv": Xp, "price": f.market(t, "price", "co2"),
              "co2": f.market(t, "co2", "price")}
        out = []
        for tg in TARGETS:
            X = Xs[tg]
            if self.family == "seasonal_naive":
                p = seasonal_naive(X, A.FEATURE_NAMES[tg])
            else:
                m = self.fam_models[tg]
                if isinstance(m, list):
                    p = predict_buckets(m, X, A._K)
                    r = self.fam_models.get(tg + "_ridge")
                    if r is not None:
                        p = np.where(np.isnan(p), predict_ridge(r, X, A._K), p)
                else:
                    p = np.asarray(predict_any(m, X, A._K), dtype=float)
            out.append(p)
        return tuple(out)


def simulate_family(artifact, df_env, family, models, params=None):
    art = dict(artifact)
    art["params"] = dict(artifact.get("params", {}))
    art["params"].update(params or {})
    ag = ModelAgent(artifact=art)
    assert ag.ready
    ag.set_family(family, models)
    return run_hard(ag, df_env, desc=f"sim {family}")


class OracleAgent:
    """Cota superior: mateix LP amb coneixement perfecte de tota la serie."""

    def __init__(self, df_env, spec=None):
        spec = spec or SPEC_HARD
        self.plan = oracle_plan(df_env, spec)
        self.i = 0
        self.cap = spec["capacity_kwh"]; self.pmax = spec["power_kw"]

    def act(self, obs):
        a = float(self.plan[self.i]); self.i += 1
        eta = 0.9 ** 0.5; soc = float(obs["soc_kwh"]); eps = 1e-4
        mc = max(0.0, min(self.pmax, (self.cap - soc) / eta - eps))
        md = max(0.0, min(self.pmax, soc * eta - eps))
        return float(max(-md, min(mc, a)))


def oracle_plan(df_env, spec):
    n = len(df_env)
    eta = spec["efficiency"] ** 0.5
    cap = spec["capacity_kwh"]; pmax = spec["power_kw"]; soc0 = spec["soc_init"] * cap
    net = (df_env["load_kw"] - df_env["pv_kw"]).values
    co2_ref, price_ref = baseline_refs(df_env)
    w = 0.7 * df_env["co2_kg_per_kwh"].values / co2_ref + 0.3 * df_env["precio_eur_per_kwh"].values / price_ref
    # variables: c (n), d (n), g (n), s (n)
    Aeq = lil_matrix((n, 4 * n)); beq = np.zeros(n)
    Aub = lil_matrix((n, 4 * n)); bub = -net
    for i in range(n):
        Aeq[i, 3 * n + i] = 1.0; Aeq[i, i] = -eta; Aeq[i, n + i] = 1.0 / eta
        if i > 0:
            Aeq[i, 3 * n + i - 1] = -1.0
        else:
            beq[i] = soc0
        Aub[i, i] = 1.0; Aub[i, n + i] = -1.0; Aub[i, 2 * n + i] = -1.0
    c = np.concatenate([np.full(2 * n, 1e-6), w, np.zeros(n)])
    bounds = [(0, pmax)] * (2 * n) + [(0, None)] * n + [(0, cap)] * n
    res = linprog(c, A_ub=Aub.tocsr(), b_ub=bub, A_eq=Aeq.tocsr(), b_eq=beq, bounds=bounds, method="highs")
    assert res.status == 0, res.message
    return res.x[:n] - res.x[n:2 * n]


def leakage_test(arrays, cs_table, n_trials=40, seed=0):
    """Test de no-filtracio: pertorba TOT el futur no permes (telemetria >= t, mercat >= t+3)
    i comprova que les features de l'agent no canvien ni un bit."""
    rng = np.random.default_rng(seed)
    idx = A.full_index()
    cal = A.calendar_matrix(idx)
    cs = A.clear_sky_series(cs_table, idx)
    ok_t = np.flatnonzero(~np.isnan(arrays["price"]))
    ok_t = ok_t[(ok_t > 400) & (ok_t < A.N_HOURS - 60)]
    worst = 0.0
    for t in rng.choice(ok_t, n_trials, replace=False):
        f0 = A.Features(arrays, cal, cs)
        pert = {k: v.copy() for k, v in arrays.items()}
        for k in A.TELEMETRY:
            pert[k][t:] = rng.normal(size=A.N_HOURS - t) * 1e3
        for k in ("price", "co2"):
            pert[k][t + 3:] = rng.normal(size=A.N_HOURS - t - 3) * 1e3
        f1 = A.Features(pert, cal, cs)
        for target in TARGETS:
            a, b = f0.build(target, int(t)), f1.build(target, int(t))
            d = np.nanmax(np.abs(np.where(np.isnan(a) & np.isnan(b), 0.0, a - b)))
            if not (np.isnan(a) == np.isnan(b)).all():
                d = np.inf
            worst = max(worst, float(d))
    return worst

In [ ]:
import sys, time, json, zipfile, warnings
from pathlib import Path
import numpy as np, pandas as pd, joblib
warnings.filterwarnings("ignore")
sys.path.insert(0, "hm_build")
for m in ("environment", "agent", "data_prep", "hard_env", "train_lib"):
    sys.modules.pop(m, None)
import agent as A, data_prep as D, hard_env as HE, train_lib as T
from environment import CampusBatteryEnv, run_agent_simulation
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
USE_GPU = T.gpu_available()
if FAST:
    FAMILIES = [f for f in FAMILIES if f not in ("catboost", "random_forest")]
print("GPU:", USE_GPU, "| data:", DATA_DIR)

## 2. Preprocessat (PV ×12) i cobertura de dades

* `build_hourly`: sèrie neta per a *features* (els `0.0` del comptador = caiguda de sensor → `NaN`).
* `build_env_frame`: taula «oficial» del simulador, calcada de `starter_kit/tutorial.ipynb` (els `0.0` es conserven, `dropna`), amb PV ×12.

In [ ]:
hourly = D.build_hourly(DATA_DIR)          # features
env_df = D.build_env_frame(DATA_DIR)       # veritat del simulador (df_hard)
cov = hourly.notna().groupby(hourly.index.to_period("M")).sum()
print("Hores amb dada per mes (novembre 2025 sense comptador/inversor/meteo; inversor fins 16/08/2026):")
display(cov[["precio_eur_per_kwh", "co2_kg_per_kwh", "load_kw", "pv_kw", "irradiancia_wm2"]])
print("vent_ms sempre 0:", float(hourly["vent_ms"].abs().max()) == 0.0,
      "| hores env:", len(env_df), "| net<0 (excedent solar):", f"{((env_df.load_kw - env_df.pv_kw) < 0).mean():.1%}")

### 2.1 Comparació d'estratègies d'imputació
Amaguem valors coneguts de `load_kw` (forats curts d'1–3 h i forats llargs de 48 h) i mesurem l'error de cada estratègia. Regla resultant: interpolació només per a forats ≤ 3 h; mai a través de forats llargs (novembre, setmanes ocultes), que es deixen a `NaN` per als lags.

In [ ]:
rng = np.random.default_rng(0)
s = hourly["load_kw"].copy()
known = np.flatnonzero(s.notna().values)
rows = []
for name, glen, n in (("forat curt (1-3 h)", (1, 3), 300), ("forat llarg (48 h)", (48, 48), 12)):
    hid = s.copy(); idxs = []
    for st in rng.choice(known[200:-200], n, replace=False):
        L = int(rng.integers(glen[0], glen[1] + 1)); hid.iloc[st:st + L] = np.nan; idxs.extend(range(st, st + L))
    idxs = np.array(sorted(set(idxs))); truth = s.iloc[idxs]
    cands = {"interpolacio lineal (sense limit)": hid.interpolate(limit_area="inside"),
             "interpolacio <= 3 h": T.interpolate_short(hid, 3),
             "mediana mes/dia-setmana/hora": T.impute_profile(hid)}
    for k, v in cands.items():
        p = v.iloc[idxs]; ok = p.notna() & truth.notna()
        rows.append({"cas": name, "estrategia": k, "MAE_kW": round(float((p[ok] - truth[ok]).abs().mean()), 1) if ok.any() else np.nan,
                     "cobertura": f"{ok.sum() / max(1, truth.notna().sum()):.0%}"})
display(pd.DataFrame(rows))

## 3. Features, calendari i test de no-filtració
Les features les construeix **el mateix codi** de `agent.py` (`agent.Features`) offline i online. El test pertorba tot el futur no permès (telemetria ≥ t, mercat ≥ t+3) i exigeix diferència exactament 0.

In [ ]:
arr = T.to_arrays(hourly)                                  # historic complet (NaN als forats)
cal = pd.DataFrame(A.calendar_matrix(hourly.index), index=hourly.index, columns=A.CAL_NAMES)
print("Hores marcades per flag:"); display(cal[[c for c in A.CAL_NAMES if c.startswith("is_")]].sum().astype(int).to_frame("hores").T)
val_df, blocks = T.make_validation_frame(env_df, hourly)
arr_tr = T.mask_blocks(arr, blocks)                        # train sense els blocs de validacio
cs_tr = T.clear_sky_table(arr_tr["pv"]); cs_full = T.clear_sky_table(arr["pv"])
leak = T.leakage_test(arr, cs_full, n_trials=40)
print("Test de leakage (max |diferencia| de features en pertorbar el futur):", leak); assert leak == 0.0
for tg in T.TARGETS: print(f"{tg:6s} {len(A.FEATURE_NAMES[tg])} features")

## 4. El nostre conjunt de test: 12 blocs × 168 h en Mode Difícil

Per a cada mes, la **darrera setmana disponible a train** (la immediatament anterior a la setmana oculta), més un bloc **sintètic de novembre 2025** (preu i CO₂ reals; consum/solar/meteo = mitjana de dues setmanes reals d'octubre i desembre). Es concatenen sense reiniciar el SOC, igual que el test ocult.

In [ ]:
display(pd.DataFrame([(s, e, len(val_df.loc[s:e])) for s, e in blocks], columns=["inici", "fi", "hores"]))
env_tr = env_df.drop(val_df.index, errors="ignore")
for s_, e_ in T.NOV_SOURCES: env_tr = env_tr.drop(env_tr.loc[s_:e_].index)   # fonts del novembre sintetic
SPEC_HARD = HE.SPEC_HARD; print(SPEC_HARD)
ORACLE = "oracle (LP amb futur perfecte, cota superior)"
ref = {ORACLE: T.run_hard(T.OracleAgent(val_df), val_df),
       "heuristica de reserva (sense model)": T.run_hard(A.BatteryAgent(artifact={}), val_df)}
display(pd.DataFrame(ref).T[["score", "co2_estalviat_pct", "cost_estalviat_pct", "infraccions"]])

## 5. Comparació de famílies de models — error de predicció (MAE)
Tots s'entrenen **sense** els blocs de validació i s'avaluen sobre ells (horitzons k=0..23 per a consum i solar, k=3..23 per a preu i CO₂).

In [ ]:
n_rounds = 150 if FAST else 400
bench, fam_models = [], {f: {} for f in FAMILIES}
for tg in T.tqdm(T.TARGETS, desc="objectius"):
    res, models = T.benchmark_target(tg, arr_tr, arr, cs_tr, blocks, FAMILIES, use_gpu=USE_GPU, n_rounds=n_rounds)
    bench.append(res)
    for f in FAMILIES: fam_models[f][tg] = models[f]
    if "ridge" in models and "lightgbm_buckets" in models:
        fam_models["lightgbm_buckets"][tg + "_ridge"] = models["ridge"]     # k>=12 sense arbres
bench = pd.concat(bench, ignore_index=True)
display(bench.pivot(index="model", columns="target", values="MAE").loc[FAMILIES, list(T.TARGETS)].round(4))
display(bench.round(4))

## 6. Comparació de famílies — puntuació oficial en Mode Difícil (el que compta)
Mateix control i mateix simulador per a tothom; només canvia qui fa les prediccions. `blend` = LightGBM per grups d'horitzó + Ridge (el que s'exporta per defecte). Trieu a `CHOSEN` quin voleu exportar (`blend`, `lgb` o `ridge` són els desplegables dins del límit de latència).

In [ ]:
models_val = T.train_final_models(arr_tr, cs_tr, cap_qs=(0.1, 0.2, 0.3))
sigma = T.residual_sigma(models_val, arr, cs_tr, blocks)       # error fora de mostra per horitzo -> DP estocastic
art_val = T.make_artifact(arr_tr, cs_tr, models_val, env_tr, sigma=sigma)
print("sigma consum (kW) k=0..5:", np.round(sigma["load"][:6]), "| sigma solar (fraccio de cel clar):", np.round(sigma["pv_frac"][:6], 2))
rows = {}
for fam in T.tqdm(FAMILIES, desc="families (simulacio Mode Dificil)"):
    rows[fam] = T.simulate_family(art_val, val_df, fam, fam_models[fam])
for pref in T.tqdm(("blend", "lgb", "ridge"), desc="desplegables"):
    rows[f"DESPLEGABLE: {pref}"] = T.simulate(art_val, val_df, {"prefer": pref}, desc=f"sim {pref}")
cmp_ctrl = pd.DataFrame(rows).T[["score", "co2_estalviat_pct", "cost_estalviat_pct", "infraccions", "ms_per_step_mean"]]
cmp_ctrl["% de l'oracle"] = (100 * cmp_ctrl["score"].astype(float) / ref[ORACLE]["score"]).round(1)
display(cmp_ctrl.sort_values("score", ascending=False))
print("Model escollit per exportar: CHOSEN =", CHOSEN)

## 7. Ajust d'hiperparàmetres amb Optuna

**7.1 LightGBM** (per objectiu): cerca bayesiana TPE sobre `learning_rate`, `num_leaves`, `min_data_in_leaf`, `feature_fraction`, `bagging_fraction`, `lambda_l2` i nombre d'arbres (limitat a 300 per la latència), minimitzant el MAE fora de mostra. Com que menys error no sempre vol dir més punts (*decision-focused learning*, Depoortere et al. 2026), els models ajustats només s'adopten si **no empitjoren la puntuació oficial**.

In [ ]:
lgb_best, lgb_log = {}, []
if N_TRIALS_LGB > 0:
    for tg in T.TARGETS:
        bp, study = T.optuna_tune_lgb(tg, arr_tr, arr, cs_tr, blocks, n_trials=(5 if FAST else N_TRIALS_LGB))
        lgb_best[tg] = bp
        lgb_log.append({"objectiu": tg, "MAE per defecte": study.trials[0].value, "MAE Optuna": study.best_value, **bp})
    display(pd.DataFrame(lgb_log).round(4))
    models_tuned = T.train_final_models(arr_tr, cs_tr, lgb_params=lgb_best, cap_qs=(0.1, 0.2, 0.3))
    sigma_tuned = T.residual_sigma(models_tuned, arr, cs_tr, blocks)
    art_tuned = T.make_artifact(arr_tr, cs_tr, models_tuned, env_tr, sigma=sigma_tuned)
    r0 = T.simulate(art_val, val_df, {"prefer": CHOSEN}); r1 = T.simulate(art_tuned, val_df, {"prefer": CHOSEN})
    print(f"Puntuacio Mode Dificil -> LightGBM per defecte: {r0['score']} ({r0['ms_per_step_mean']} ms/pas) | Optuna: {r1['score']} ({r1['ms_per_step_mean']} ms/pas)")
    USE_TUNED_LGB = r1["score"] >= r0["score"] and r1["infraccions"] == 0
    if USE_TUNED_LGB: art_val, sigma = art_tuned, sigma_tuned
    else: lgb_best = {}
    print("S'adopten els hiperparametres d'Optuna:", USE_TUNED_LGB)

**7.2 Control** (aproximació paramètrica de la funció de cost ajustada sobre el simulador; Ghadimi & Powell 2023): `sigma_scale` (pes de l'error de predicció al DP estocàstic), `cap_q` (quantil del sostre de descàrrega), `buf_kw`, `gamma`, `w_co2`, `net_bias_kw` i els pesos de la mescla LightGBM/Ridge per objectiu. Objectiu = puntuació oficial sobre els 12 blocs.

**Antisobreajust:** 12 setmanes són poques per a 10 paràmetres. Una configuració només s'accepta si millora la de per defecte a **les dues meitats** dels blocs (parells i senars) i guanya ≥ 0,05 punts en total; si no, es mantenen els valors per defecte.

In [ ]:
best = {"prefer": CHOSEN}
if N_TRIALS_CTRL > 0:
    best, study_ctrl, tab_ctrl = T.optuna_tune_control(art_val, val_df, blocks, n_trials=(8 if FAST else N_TRIALS_CTRL), base={"prefer": CHOSEN})
    print("Assaig escollit:", tab_ctrl.attrs["chosen_trial"], "(0 = valors per defecte)")
    display(tab_ctrl.head(10).round(3))
    try:
        import optuna
        display(pd.Series(optuna.importance.get_param_importances(study_ctrl)).round(3).to_frame("importancia").T)
    except Exception as ex:
        print("sense importancies:", ex)
print("Parametres escollits:", best)

## 8. Entrenament final amb totes les dades i exportació

In [ ]:
cq = best.get("cap_q", A.DEFAULT_PARAMS["cap_q"])
models_full = T.train_final_models(arr, cs_full, lgb_params=lgb_best, cap_qs=(cq,) if cq else ())
art_final = T.make_artifact(arr, cs_full, models_full, env_df, params=best, sigma=sigma)   # sigma mesurada fora de mostra
joblib.dump(art_final, "hm_build/model_bateria.pkl", compress=3)
with zipfile.ZipFile("hm_build/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("hm_build/agent.py", "agent.py"); z.write("hm_build/model_bateria.pkl", "model_bateria.pkl")
for f in ("model_bateria.pkl", "submission.zip"):
    Path(f).write_bytes(Path("hm_build", f).read_bytes())            # copia al costat del notebook
print({f: f"{Path('hm_build', f).stat().st_size / 1e6:.2f} MB" for f in ("model_bateria.pkl", "submission.zip")})
print("params:", art_final["params"])

## 9. Validació final en Mode Difícil

* **Validació honesta**: 12 blocs amb l'artefacte entrenat *sense* aquests blocs.
* **Train complet** amb l'artefacte final (in-sample: serveix per comprovar infraccions i robustesa, no com a estimació).
* **Entorn oficial sense modificar** (`CampusBatteryEnv(df_hard, spec=spec_hard)`), que no retalla l'horitzó ni dona meteo: comprova que l'agent funciona igualment.
* **Novembre** sol (arrencada en fred) i latència per pas.

In [ ]:
final = {}
final["validacio 12 blocs (honesta)"] = T.simulate(art_val, val_df, best)
final["train complet (in-sample)"] = T.simulate(art_final, env_df)
ag = A.BatteryAgent(artifact=art_final)
r = run_agent_simulation(ag, env_df.drop(columns=list(HE.METEO_KEYS)), spec=SPEC_HARD); r["fallbacks"] = ag.n_fallback
final["entorn oficial sense modificar (train)"] = r
s, e = [b for b in blocks if b[0].month == 11][0]
final["nomes novembre (cold start)"] = T.simulate(art_val, val_df.loc[s:e], best)
ag = A.BatteryAgent(); ag_ok = ag.ready                      # carrega hm_build/model_bateria.pkl des de disc
final["submission (agent.py + pkl des de disc, in-sample)"] = {**T.run_hard(ag, val_df), "fallbacks": ag.n_fallback}
fin = pd.DataFrame(final).T
display(fin[[c for c in ["score", "co2_estalviat_pct", "cost_estalviat_pct", "infraccions", "fallbacks", "ms_per_step_mean", "ms_per_step_max"] if c in fin.columns]])
assert ag_ok and (fin["infraccions"] == 0).all() and (fin["fallbacks"] == 0).all()
print("OK: zero infraccions i zero passos de reserva en totes les simulacions.")

In [ ]:
# Puntuacio per bloc de validacio (mes a mes)
res, env = T.simulate(art_val, val_df, best, return_env=True)
ts = pd.DatetimeIndex(env.df["Fecha"]); rows = []
for s, e in blocks:
    m = np.asarray((ts >= s) & (ts <= e)); r = env.evaluate_results(mask=m)
    rows.append({"bloc": s.strftime("%Y-%m"), "score": r["score"], "co2_%": r["co2_estalviat_pct"], "cost_%": r["cost_estalviat_pct"], "infraccions": r["infraccions"]})
display(pd.DataFrame(rows).set_index("bloc").T)
try:
    import matplotlib.pyplot as plt
    i0 = int(np.flatnonzero(ts >= blocks[7][0])[0]); sl = slice(i0, i0 + 168)
    fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    ax[0].plot(ts[sl], env.df["load_kw"].values[sl] - env.df["pv_kw"].values[sl], label="consum net (kW)")
    ax[0].plot(ts[sl], np.array(env.actions)[sl], label="consigna bateria (kW)"); ax[0].axhline(0, c="k", lw=.5); ax[0].legend()
    ax[1].plot(ts[sl], np.array(env.soc_series)[sl], c="tab:green", label="SOC (kWh)"); ax2 = ax[1].twinx()
    ax2.plot(ts[sl], env.df["co2_kg_per_kwh"].values[sl], c="tab:red", alpha=.5, label="CO2"); ax[1].legend(loc="upper left"); ax2.legend(loc="upper right")
    plt.tight_layout(); plt.show()
except Exception as ex:
    print("sense grafic:", ex)

## 10. Verificació del lliurament
Executa `check_submission.py` oficial si es troba (repo local); a Kaggle es fa una comprovació equivalent mínima.

In [ ]:
chk = [p for p in (Path("../starter_kit/check_submission.py"), Path("starter_kit/check_submission.py")) if p.exists()]
if chk:
    out = subprocess.run([sys.executable, str(chk[0].resolve()), str(Path("hm_build/submission.zip").resolve())], capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(out.stdout[-1500:], out.stderr[-500:])
else:
    import ast
    bad = {"os", "sys", "shutil", "subprocess", "socket", "urllib", "requests", "http", "importlib", "inspect", "threading", "multiprocessing", "ctypes"}
    tree = ast.parse(Path("hm_build/agent.py").read_text(encoding="utf-8"))
    imps = {a.name.split(".")[0] for n in ast.walk(tree) if isinstance(n, ast.Import) for a in n.names} | {n.module.split(".")[0] for n in ast.walk(tree) if isinstance(n, ast.ImportFrom) and n.module}
    print("imports:", sorted(imps), "| prohibits:", sorted(imps & bad))
print("Fitxers a lliurar: hm_build/submission.zip  (agent.py + model_bateria.pkl)")